# Agent 1: Binary Header-Only Naive Bayes Email Classifier
### A Regularized, Calibrated, Nested-CV Evaluated Component for Multi-Agent Phishing Detection

---

### Project Overview and Binary Architecture
This notebook implements **Agent 1**, the email header analysis specialist in our binary multi-agent anti-phishing firewall (`0 = Safe`, `1 = Not Safe` combining spam and phishing):
* **Agent 1 (This Notebook):** A genuinely two-class ($K=2$) regularized Naive Bayes classifier operating strictly on email metadata and header fields (`sender`, `receiver`, `date`, `subject`). It outputs calibrated `nb_score = P(Not Safe)` in $[0, 1]$ and clipped log-odds `nb_logit`.
* **Agent 2:** A fine-tuned DistilBERT Transformer operating on subject + body text, exporting `transformer_scores.csv` (`email_id, split, text_score`) for `val` and `test` rows.
* **Agent 3:** A Logistic Regression stacking meta-classifier that merges Agent 1 and Agent 2 scores on `email_id`, fits on `val` rows, and evaluates once on `test` rows.

### Scope and Isolation Guarantee
* Agent 1 has access **only** to `sender`, `receiver`, `date`, and `subject`. It **never inspects the email body**, body URLs, or attachments.
* `val` and `test` labels are **never used** for training, hyperparameter selection, temperature calibration, or threshold tuning. (Note: `test` was inspected in earlier project iterations and is not a pristine blind hold-out.)
* Metadata (`email_id`, `split`, `source`) are strictly excluded from predictive features.

### Evaluation Hierarchy
1. **Nested Sender-Disjoint CV:** Primary internal performance estimate for generalization to unseen senders.
2. **Template-Disjoint / Template-Purged Diagnostics:** Stress tests measuring performance when subject templates are novel.
3. **Leave-One-Source-Out (LOSO):** Diagnostic measuring cross-corpus shift (comparable to Agent 2 on `CEAS_08` and `SpamAssasin`).
4. **Validation Split (Reporting Only):** Evaluated once on the frozen model; never used for tuning.

In [5]:
# 2. Imports and configuration
import os, sys, glob, json, hashlib, pathlib, subprocess, warnings, random, time, platform
import numpy as np, pandas as pd, joblib, sklearn, scipy
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, log_loss, average_precision_score, accuracy_score,
                             precision_recall_curve, confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------ reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# ------------------------------------------------------------------ experiment switches
N_SPLITS = 5                      # outer folds / final grouped CV
N_INNER  = 4                      # inner folds used inside every outer training set
QUICK_SEARCH = os.environ.get("AGENT1_QUICK", "0") == "1"
NESTED_GRID_MODE = "full"         # 96-candidate grid searched INDEPENDENTLY inside every outer fold
N_JOBS = int(os.environ.get("AGENT1_JOBS", max(1, min(4, os.cpu_count() or 1))))

RUN_TEMPLATE_EXPERIMENTS = True   # section 9
RUN_REDUCED_COMPARISON = True     # section 13: reduced vs full nested grid
GENERATE_TRAIN_OOF = False        # optional: Agent 3 fits on val, so train OOF is not required by default
OOF_PURGE_TEMPLATES = True        # used only if GENERATE_TRAIN_OOF=True
USE_CACHE = True                  # checkpoint finished outer folds in OUT/cache
RUN_LOSO   = True                 # section 10
RUN_ABLATION = True               # section 11

# binary decision rule for `pred_class` ("argmax" | "f1_opt" | "macro_f1" | "recall_at_precision")
DECISION_MODE = "f1_opt"
MIN_NOTSAFE_PRECISION = 0.70
PRECISION_FLOORS = (0.60, 0.65, 0.70, 0.75, 0.80)
FINAL_CFG_OVERRIDE = None
REPORT_ON_VAL = True
REPORT_ON_TEST = False
OUT = pathlib.Path("agent1_binary_outputs"); OUT.mkdir(exist_ok=True); (OUT / "cache").mkdir(exist_ok=True)
HEADER_COLS = ["sender", "receiver", "date", "subject"]
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_colwidth", 60)
print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__} | scipy {scipy.__version__}")
print(f"QUICK_SEARCH={QUICK_SEARCH} NESTED_GRID_MODE={NESTED_GRID_MODE} N_JOBS={N_JOBS} DECISION_MODE={DECISION_MODE}")

python 3.13.18 | numpy 2.1.3 | pandas 2.3.3 | sklearn 1.6.1 | scipy 1.16.3
QUICK_SEARCH=False NESTED_GRID_MODE=full N_JOBS=4 DECISION_MODE=f1_opt


## 1. Model Architecture and Embedded Binary Inference Module

This section encapsulates the binary Agent 1 model within a self-contained source string (`HEADER_SRC`), written to `agent1_binary_outputs/agent1_header.py` at runtime so that:
1. Multiprocessing workers (`joblib.Parallel`) import the exact same classes across subprocesses.
2. The serialized artifact (`agent1_binary_model.joblib`) can be loaded in a fresh Python process without notebook state.

All header parsing, 36 engineered features, abstention/missing-value handling (B2–B5, N1, Q1–Q2), `feat_bound`, `lik_scale`, group weighting, and subject-token Naive Bayes are preserved, refactored natively for **$K = 2$** (`0 = safe`, `1 = not_safe`) with log-odds temperature scaling and a single binary log-odds decision threshold.

In [6]:
# 3. Agent 1 binary model code (feature extractor + K=2 Naive Bayes). Everything lives in this notebook.
HEADER_SRC = '"""\nagent1_header.py - Binary Header-only Naive Bayes agent (v8.0.0-binary)\n\nInputs : sender, receiver, date, subject (raw header strings). Never the body.\nOutputs: P(safe), P(not_safe) in fixed order (0=safe, 1=not_safe).\n         nb_score = P(not_safe) in [0, 1].\n"""\nimport re, datetime\nfrom email.header import decode_header, make_header\nfrom email.utils import getaddresses, parsedate_tz\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.optimize import minimize_scalar\n\n__version__ = "8.0.0-binary"\n\nCLASS_NAMES = ("safe", "not_safe")\nN_CLASSES = 2\nPROB_FLOOR = 1e-6\nLOGIT_CLIP = 10.0\nMAX_RAW = 4096\nMAX_HDR = 512\n\n_PREFIX_RE = re.compile(r"^\\s*(?:re|fwd?|fw|antw|wg|tr|aw)\\s*[:\\-]\\s*", re.I)\n_ADDR_RE = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+")\n_IPV4_RE = re.compile(r"^\\d{1,3}(?:\\.\\d{1,3}){3}$")\n_TZ_TAIL_RE = re.compile(r"\\b([A-Z]{1,4})\\b\\s*$")\n_TZ_NUM_RE = re.compile(r"\\d:\\d{2}(?::\\d{2})?\\s*([+-]\\d{4})\\b")\n\nFREE_MAIL = frozenset({\n    "gmail.com", "yahoo.com", "hotmail.com", "aol.com", "outlook.com",\n    "mail.com", "msn.com", "live.com", "ymail.com", "comcast.net",\n    "sbcglobal.net", "verizon.net", "att.net", "bellsouth.net",\n    "rediffmail.com", "cox.net", "charter.net", "shaw.ca", "earthlink.net"})\n\nBRAND_TOKENS = frozenset({\n    "paypal", "ebay", "amazon", "apple", "microsoft", "google",\n    "chase", "wellsfargo", "bankofamerica", "citi", "citigroup",\n    "barclays", "hsbc", "netflix", "irs", "usps", "fedex", "ups",\n    "dhl", "facebook", "instagram", "whatsapp", "linkedin", "twitter"})\nFUZZY_BRANDS = frozenset(b for b in BRAND_TOKENS if len(b) >= 5)\nCOMMON_WORDS = frozenset({"apply", "ample", "maple", "grape", "phase", "chose", "chasm", "apples"})\n\nSUSPICIOUS_TLDS = frozenset({\n    "tk", "ml", "ga", "cf", "gq", "xyz", "top", "work", "loan",\n    "click", "fit", "surf", "buzz", "racing", "date", "faith"})\n\nKNOWN_TWO_LEVEL_TLDS = frozenset({\n    "co.uk", "org.uk", "me.uk", "ltd.uk", "plc.uk", "net.uk",\n    "com.au", "net.au", "org.au", "edu.au", "gov.au",\n    "co.nz", "net.nz", "org.nz", "ac.nz", "govt.nz",\n    "co.za", "org.za", "net.za", "gov.za",\n    "co.jp", "ne.jp", "or.jp", "go.jp", "ac.jp",\n    "com.br", "net.br", "org.br", "gov.br",\n    "com.mx", "org.mx", "net.mx", "edu.mx", "gob.mx",\n    "com.cn", "net.cn", "org.cn", "gov.cn", "edu.cn",\n    "co.in", "net.in", "org.in", "gen.in", "firm.in",\n    "gc.ca", "fed.us"})\n\nRFC_TZ_OFFSETS = {\n    "UT": 0, "UTC": 0, "GMT": 0, "Z": 0,\n    "EDT": -4, "EST": -5, "CDT": -5, "CST": -6,\n    "MDT": -6, "MST": -7, "PDT": -7, "PST": -8,\n    "A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6,\n    "G": 7, "H": 8, "I": 9, "K": 10, "L": 11, "M": 12,\n    "N": -1, "O": -2, "P": -3, "Q": -4, "R": -5, "S": -6,\n    "T": -7, "U": -8, "V": -9, "W": -10, "X": -11, "Y": -12}\n\n\ndef _decode(v, limit=MAX_HDR):\n    if isinstance(v, bytes):\n        v = v.decode("utf-8", "replace")\n    if not isinstance(v, str):\n        return ""\n    raw = v[:MAX_RAW]\n    try:\n        dec = str(make_header(decode_header(raw)))\n    except Exception:\n        dec = raw\n    return dec[:limit].strip()\n\n\ndef _is_ip(host):\n    return bool(_IPV4_RE.match(str(host).strip("[] ")))\n\n\ndef registrable_domain(domain):\n    if not domain or not isinstance(domain, str):\n        return ""\n    d = domain.strip().lower().strip(". >)(<[]")\n    if _IPV4_RE.match(d):\n        return d\n    parts = d.split(".")\n    if len(parts) <= 2:\n        return d\n    if ".".join(parts[-2:]) in KNOWN_TWO_LEVEL_TLDS:\n        return ".".join(parts[-3:])\n    return ".".join(parts[-2:])\n\n\ndef _normalize_subject_text(dec):\n    txt = dec.lower()\n    for _ in range(5):\n        new = _PREFIX_RE.sub("", txt).strip()\n        if new == txt:\n            break\n        txt = new\n    txt = re.sub(r"\\d+", "#", txt)\n    txt = re.sub(r"[^\\w\\s#]", " ", txt)\n    return " ".join(txt.split())\n\n\ndef normalize_subject(s):\n    return _normalize_subject_text(_decode(s))\n\n\ndef _parse_addrs(text):\n    out = []\n    if not text or not isinstance(text, str):\n        return out\n    try:\n        try:\n            items = getaddresses([text], strict=False)\n        except TypeError:\n            items = getaddresses([text])\n        out = [(n.strip(), a.strip().lower()) for n, a in items if a and "@" in a]\n    except Exception:\n        out = []\n    if not out:\n        out = [("", a.lower()) for a in _ADDR_RE.findall(text)]\n    return out\n\n\ndef _within_one_edit(a, b):\n    if a == b:\n        return False\n    la, lb = len(a), len(b)\n    if abs(la - lb) > 1:\n        return False\n    if la == lb:\n        return sum(c1 != c2 for c1, c2 in zip(a, b)) == 1\n    short, long_ = (a, b) if la < lb else (b, a)\n    return any(long_[:i] + long_[i + 1:] == short for i in range(len(long_)))\n\n\ndef canonical_sender_identity(dom, addr):\n    if not addr or "@" not in str(addr):\n        return None\n    d = str(dom or "").strip().lower()\n    if not d or d in FREE_MAIL or _is_ip(d):\n        return f"addr:{str(addr).strip().lower()}"\n    return f"dom:{d}"\n\n\ndef _ensure_df(x):\n    if isinstance(x, pd.DataFrame):\n        return x\n    if isinstance(x, dict):\n        return pd.DataFrame([x])\n    if isinstance(x, (list, tuple)):\n        return pd.DataFrame(list(x))\n    raise TypeError(type(x))\n\n\n_SENDER_EMPTY = dict(\n    s_missing=1, s_addr="", s_dom="", s_tld="", s_dom_levels=0, s_has_num=0, s_has_hyphen=0,\n    s_is_freemail=0, s_suspicious_tld=0, s_brand_in_domain=0, s_brand_spoof_host=0, s_typo_brand=0,\n    s_ip_domain=0, s_punycode=0, s_disp_len=0, s_disp_has_excl=0, s_disp_has_quest=0,\n    s_disp_has_at=0, s_disp_has_dom=0)\n\n\ndef _sender_info(v):\n    row = dict(_SENDER_EMPTY)\n    addrs = _parse_addrs(_decode(v))\n    if not addrs:\n        return row\n    disp, a = addrs[0]\n    host = a.split("@", 1)[1].strip(".[]> <)(") if "@" in a else ""\n    if not host:\n        return row\n    row["s_missing"] = 0\n    row["s_addr"] = a\n    labels = host.split(".")\n    row["s_dom_levels"] = len(labels)\n    row["s_disp_len"] = len(disp)\n    row["s_disp_has_excl"] = int("!" in disp)\n    row["s_disp_has_quest"] = int("?" in disp)\n    row["s_disp_has_at"] = int("@" in disp)\n    row["s_disp_has_dom"] = int(bool(re.search(r"\\b[a-z0-9-]+\\.[a-z]{2,}\\b", disp, re.I)))\n    row["s_punycode"] = int(any(l.startswith("xn--") for l in labels))\n    if _is_ip(host):\n        row["s_dom"] = host.strip("[]")\n        row["s_ip_domain"] = 1\n        return row\n    reg = registrable_domain(host)\n    tld = reg.split(".")[-1] if "." in reg else ""\n    row["s_dom"], row["s_tld"] = reg, tld\n    row["s_has_num"] = int(bool(re.search(r"\\d", reg)))\n    row["s_has_hyphen"] = int("-" in reg)\n    row["s_is_freemail"] = int(reg in FREE_MAIL)\n    row["s_suspicious_tld"] = int(tld in SUSPICIOUS_TLDS)\n    reg_tokens = {t for t in re.split(r"[^a-z0-9]+", reg) if t}\n    sub_part = host[: len(host) - len(reg)] if host.endswith(reg) else ""\n    sub_tokens = {t for t in re.split(r"[^a-z0-9]+", sub_part) if t}\n    in_reg = reg_tokens & BRAND_TOKENS\n    row["s_brand_in_domain"] = int(bool(in_reg))\n    row["s_brand_spoof_host"] = int(bool((sub_tokens & BRAND_TOKENS) - in_reg))\n    row["s_typo_brand"] = int(any(\n        len(t) >= 5 and t not in COMMON_WORDS and t not in BRAND_TOKENS and _within_one_edit(t, b)\n        for t in reg_tokens for b in FUZZY_BRANDS))\n    return row\n\n\ndef _receiver_info(v, sdom):\n    dec = _decode(v)\n    addrs = _parse_addrs(dec)\n    undis = int("undisclosed" in dec.lower())\n    row = dict(r_missing=0, r_state=0, r_count=0, r_disp_len=0, r_disp_has_at=0, r_has_undisclosed=undis,\n               r_same_domain_as_sender=0, r_is_freemail=0)\n    if not addrs:\n        row["r_missing"] = 0 if undis else 1\n        row["r_state"] = 1 if undis else 0\n        return row\n    disp, a = addrs[0]\n    host = a.split("@", 1)[1] if "@" in a else ""\n    reg = registrable_domain(host)\n    row.update(r_state=1 if undis else (2 if len(addrs) == 1 else 3),\n               r_count=len(addrs), r_disp_len=len(disp), r_disp_has_at=int("@" in disp),\n               r_same_domain_as_sender=int(reg == sdom and bool(sdom)), r_is_freemail=int(reg in FREE_MAIL))\n    return row\n\n\ndef _date_info(v):\n    dec = _decode(v)\n    if not dec:\n        return 0, -1, -1, 0\n    try:\n        pt = parsedate_tz(dec)\n    except Exception:\n        pt = None\n    if pt is None:\n        return 3, -1, -1, 0\n    y, mo, d, hr = pt[0], pt[1], pt[2], pt[3]\n    if not (0 <= hr <= 23):\n        return 3, -1, -1, 0\n    try:\n        wd = datetime.date(y, mo, d).weekday()\n    except Exception:\n        return 3, -1, -1, 0\n    body = re.sub(r"\\([^)]*\\)\\s*$", "", dec).strip()\n    if _TZ_NUM_RE.search(body):\n        return 2, hr, wd, int(round((pt[9] or 0) / 3600.0))\n    m = _TZ_TAIL_RE.search(body)\n    if m and m.group(1) in RFC_TZ_OFFSETS:\n        return 2, hr, wd, RFC_TZ_OFFSETS[m.group(1)]\n    return 1, hr, wd, 0\n\n\ndef _subject_info(v):\n    dec = _decode(v)\n    if not dec:\n        return dict(u_missing=1, u_len=0, u_num_words=0, u_caps_ratio=0.0, u_num_punct=0,\n                    u_num_digits=0, u_is_fwd=0, u_is_reply=0, u_norm="")\n    letters = [c for c in dec if c.isalpha()]\n    return dict(\n        u_missing=0, u_len=len(dec), u_num_words=len(dec.split()),\n        u_caps_ratio=(sum(c.isupper() for c in letters) / len(letters)) if letters else 0.0,\n        u_num_punct=sum(c in "!?;:#$%&\'\\"" for c in dec),\n        u_num_digits=sum(c.isdigit() for c in dec),\n        u_is_fwd=int(bool(re.search(r"^\\s*(?:fwd?|fw)\\s*[:\\-]", dec, re.I))),\n        u_is_reply=int(bool(re.search(r"^\\s*(?:re|antw|wg|tr|aw)\\s*[:\\-]", dec, re.I))),\n        u_norm=_normalize_subject_text(dec))\n\n\ndef extract_features(df):\n    df = _ensure_df(df)\n    N = len(df)\n    col = lambda name: df[name].tolist() if name in df else [None] * N\n    S = pd.DataFrame([_sender_info(v) for v in col("sender")], index=df.index)\n    R = pd.DataFrame([_receiver_info(v, d) for v, d in zip(col("receiver"), S["s_dom"].tolist())], index=df.index)\n    D = pd.DataFrame([_date_info(v) for v in col("date")], index=df.index,\n                     columns=["d_date_status", "d_hour", "d_weekday", "d_tz_h"])\n    U = pd.DataFrame([_subject_info(v) for v in col("subject")], index=df.index)\n    return pd.concat([S, R, D, U], axis=1)\n\n\nSPEC = {\n    "s_missing": ("bin",), "s_dom_levels": ("cap", 6), "s_has_num": ("bin",), "s_has_hyphen": ("bin",),\n    "s_is_freemail": ("bin",), "s_suspicious_tld": ("bin",), "s_brand_in_domain": ("bin",),\n    "s_brand_spoof_host": ("bin",), "s_typo_brand": ("bin",), "s_ip_domain": ("bin",), "s_punycode": ("bin",),\n    "s_disp_len": ("quant",), "s_disp_has_excl": ("bin",), "s_disp_has_quest": ("bin",),\n    "s_disp_has_at": ("bin",), "s_disp_has_dom": ("bin",), "s_tld": ("cat",),\n    "r_missing": ("bin",), "r_state": ("cap", 3), "r_count": ("cap", 5), "r_disp_len": ("quant",), "r_disp_has_at": ("bin",),\n    "r_has_undisclosed": ("bin",), "r_same_domain_as_sender": ("bin",), "r_is_freemail": ("bin",),\n    "d_date_status": ("cap", 3), "d_hour": ("edges", list(range(0, 24, 2))),\n    "d_weekday": ("cap", 6), "d_tz_h": ("edges", list(range(-12, 14, 2))),\n    "u_missing": ("bin",), "u_len": ("quant",), "u_num_words": ("quant",), "u_caps_ratio": ("quant",),\n    "u_num_punct": ("quant",), "u_num_digits": ("quant",), "u_is_fwd": ("bin",), "u_is_reply": ("bin",)}\n\nFEATURE_GROUPS = {g: [f for f in SPEC if f.startswith(p)]\n                  for g, p in (("sender", "s_"), ("receiver", "r_"), ("date", "d_"), ("subject", "u_"))}\nALL_FEATURES = [f for g in ("sender", "receiver", "date", "subject") for f in FEATURE_GROUPS[g] if f != "r_state"]\nCOMPACT_RECEIVER = ["r_state", "r_same_domain_as_sender", "r_is_freemail"]\nALL_FEATURES_COMPACT = [f for f in ALL_FEATURES if not f.startswith("r_")] + COMPACT_RECEIVER\nMISSINGNESS_FEATURES = ["s_missing", "r_missing", "r_state", "u_missing", "d_date_status"]\nCLOCK_FEATURES = ["d_hour", "d_weekday", "d_tz_h"]\n\n\nclass HeaderPreprocessor:\n    def __init__(self, features, n_bins=24, top_tld=20):\n        self.features = list(features)\n        self.n_bins, self.top_tld = n_bins, top_tld\n\n    def fit(self, Xf):\n        self.edges_, self.tlds_ = {}, []\n        M = self._masks(Xf)\n        for f in self.features:\n            kind = SPEC[f][0]\n            if kind == "quant":\n                v = pd.to_numeric(Xf[f], errors="coerce").to_numpy(dtype=float)\n                am = self.abstain_mask(f, M)\n                if am is not None:\n                    v = v[~am]\n                v = v[~np.isnan(v)]\n                self.edges_[f] = (np.unique(np.quantile(v, np.linspace(0, 1, self.n_bins + 1)[1:-1])).tolist()\n                                  if len(v) else [])\n            elif kind == "edges":\n                self.edges_[f] = list(SPEC[f][1])\n        if "s_tld" in self.features and self.top_tld > 0:\n            vc = Xf["s_tld"].astype(str).replace("nan", "").value_counts()\n            self.tlds_ = [t for t in vc.index if t][: self.top_tld]\n        self.n_categories_ = [self._k(f) for f in self.features]\n        return self\n\n    def _k(self, f):\n        kind = SPEC[f][0]\n        if kind == "bin":\n            return 3\n        if kind == "cap":\n            return SPEC[f][1] + 2\n        if kind in ("quant", "edges"):\n            return len(self.edges_[f]) + 3\n        if kind == "cat":\n            return len(self.tlds_) + 3\n        raise ValueError(kind)\n\n    def missing_category(self, f):\n        if f in ("s_missing", "r_missing", "u_missing"):\n            return 1\n        if f in ("d_date_status", "r_state"):\n            return 0\n        if SPEC[f][0] in ("quant", "edges", "cat"):\n            return self._k(f) - 2\n        return None\n\n    @staticmethod\n    def _masks(Xf):\n        n = len(Xf)\n        z = np.zeros(n, bool)\n        g = lambda c: Xf[c].to_numpy() if c in Xf.columns else None\n        s_m = g("s_missing"); s_m = s_m.astype(bool) if s_m is not None else z\n        r_m = g("r_missing"); r_m = r_m.astype(bool) if r_m is not None else z\n        u_m = g("u_missing"); u_m = u_m.astype(bool) if u_m is not None else z\n        rc = g("r_count"); r_noaddr = (rc == 0) if rc is not None else z\n        st = g("d_date_status")\n        d_bad = np.isin(st, (0, 3)) if st is not None else z\n        d_notz = (st != 2) if st is not None else z\n        return dict(s=s_m, r=r_m, u=u_m, r_noaddr=np.asarray(r_noaddr, bool),\n                    d_bad=np.asarray(d_bad, bool), d_notz=np.asarray(d_notz, bool))\n\n    @staticmethod\n    def abstain_mask(f, M):\n        if f in MISSINGNESS_FEATURES:\n            return None\n        if f.startswith("s_"):\n            return M["s"]\n        if f.startswith("r_"):\n            if f == "r_same_domain_as_sender":\n                return M["r"] | M["r_noaddr"] | M["s"]\n            return M["r"] if f in ("r_count", "r_has_undisclosed") else (M["r"] | M["r_noaddr"])\n        if f.startswith("u_"):\n            return M["u"]\n        if f.startswith("d_"):\n            return M["d_notz"] if f == "d_tz_h" else M["d_bad"]\n        return None\n\n    def transform(self, Xf):\n        M = self._masks(Xf)\n        cols = []\n        for f in self.features:\n            K = self._k(f)\n            kind = SPEC[f][0]\n            if kind == "bin":\n                c = np.clip(pd.to_numeric(Xf[f], errors="coerce").fillna(0).to_numpy(dtype=int), 0, 1)\n            elif kind == "cap":\n                c = np.clip(pd.to_numeric(Xf[f], errors="coerce").fillna(0).to_numpy(dtype=int), 0, SPEC[f][1])\n            elif kind in ("quant", "edges"):\n                ed = self.edges_[f]\n                v = pd.to_numeric(Xf[f], errors="coerce").to_numpy(dtype=float)\n                nan = np.isnan(v)\n                c = np.searchsorted(ed, np.nan_to_num(v, nan=0.0), side="right")\n                c[nan] = len(ed) + 1\n            else:\n                m = {t: i for i, t in enumerate(self.tlds_)}\n                s = Xf[f].astype(str).str.lower()\n                c = np.array([len(self.tlds_) + 1 if (not x or x == "nan") else m.get(x, len(self.tlds_))\n                              for x in s], dtype=int)\n            am = self.abstain_mask(f, M)\n            if am is not None:\n                c = np.where(am, K - 1, c)\n            cols.append(c)\n        return np.column_stack(cols).astype(np.int32)\n\n\nclass SmoothedCategoricalNB:\n    def __init__(self, alpha=1.0, lik_scale=1.0, max_missing_log_odds=1.5,\n                 feat_bound=2.5, smoothing="marginal"):\n        self.alpha, self.lik_scale = float(alpha), float(lik_scale)\n        self.max_missing_log_odds = float(max_missing_log_odds)\n        self.feat_bound = float(feat_bound) if feat_bound is not None else 0.0\n        self.smoothing = smoothing\n\n    def fit(self, X, y, n_categories, preprocessor=None, feature_weights=None):\n        y = np.asarray(y, dtype=int)\n        if not set(np.unique(y)) <= {0, 1}:\n            raise ValueError(f"Binary labels {{0, 1}} required, got {np.unique(y).tolist()}")\n        C = N_CLASSES\n        self.n_internal_ = C\n        cc = np.bincount(y, minlength=C).astype(float)\n        if (cc == 0).any():\n            raise ValueError(f"Training data must contain both classes {{0, 1}}; counts={cc.tolist()}")\n        self.log_prior_ = np.log(cc / cc.sum())\n        self.prior_probs_ = cc / cc.sum()\n        self.weights_ = np.ones(len(n_categories)) if feature_weights is None else np.asarray(feature_weights, float)\n        self.tables_ = []\n        for f, K in enumerate(n_categories):\n            col = X[:, f]\n            real = col != (K - 1)\n            cnt = np.vstack([np.bincount(col[(y == c) & real], minlength=K)[: K - 1] for c in range(C)]).astype(float)\n            n_real = cnt.sum(1)\n            tot = cnt.sum(0)\n            if self.smoothing == "laplace":\n                p = (cnt + self.alpha) / (n_real[:, None] + self.alpha * (K - 1))\n            else:\n                marg = (tot + 1.0) / (tot.sum() + (K - 1))\n                p = (cnt + self.alpha * marg) / (n_real[:, None] + self.alpha)\n            ll = np.zeros((K, C))\n            ll[: K - 1] = np.log(p).T\n            ll[: K - 1][tot == 0] = 0.0\n            if self.feat_bound > 0.0:\n                for k in range(K - 1):\n                    ll[k] = np.clip(ll[k] - np.mean(ll[k]), -self.feat_bound, self.feat_bound)\n            if self.max_missing_log_odds > 0.0 and preprocessor is not None:\n                miss = preprocessor.missing_category(preprocessor.features[f])\n                if miss is not None and miss < K - 1:\n                    ll[miss] = np.clip(ll[miss] - np.mean(ll[miss]), -self.max_missing_log_odds, self.max_missing_log_odds)\n            self.tables_.append(ll)\n        return self\n\n    def log_joint(self, X):\n        acc = np.zeros((X.shape[0], self.n_internal_))\n        for f, ll in enumerate(self.tables_):\n            acc += self.weights_[f] * ll[X[:, f]]\n        return self.log_prior_ + self.lik_scale * acc\n\n    def predict_proba(self, X):\n        z = self.log_joint(X)\n        z = z - z.max(1, keepdims=True)\n        p = np.exp(z)\n        return p / p.sum(1, keepdims=True)\n\n\nclass SubjectTokenNB:\n    def __init__(self, min_df=10, max_vocab=3000, alpha=0.5, clip=2.0):\n        self.min_df, self.max_vocab, self.alpha, self.clip = int(min_df), int(max_vocab), float(alpha), float(clip)\n\n    @staticmethod\n    def _docs(texts):\n        return [t.split() if isinstance(t, str) else [] for t in texts]\n\n    def _matrix(self, docs):\n        rows, cols = [], []\n        for i, toks in enumerate(docs):\n            for t in set(toks):\n                j = self.vocab_.get(t)\n                if j is not None:\n                    rows.append(i); cols.append(j)\n        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), len(self.vocab_)))\n\n    def fit(self, texts, y, n_classes=N_CLASSES):\n        docs = self._docs(texts)\n        df_count = {}\n        for d in docs:\n            for t in set(d):\n                df_count[t] = df_count.get(t, 0) + 1\n        items = sorted(((t, c) for t, c in df_count.items() if c >= self.min_df), key=lambda x: (-x[1], x[0]))\n        items = items[: self.max_vocab]\n        self.vocab_ = {t: i for i, (t, _) in enumerate(items)}\n        self.n_classes_ = int(n_classes)\n        V = len(self.vocab_)\n        if V == 0:\n            self.L_ = np.zeros((0, self.n_classes_))\n            return self\n        X = self._matrix(docs)\n        y = np.asarray(y, dtype=int)\n        Y = sparse.csr_matrix((np.ones(len(y)), (y, np.arange(len(y)))), shape=(self.n_classes_, len(y)))\n        cnt = np.asarray((Y @ X).todense())\n        tot = cnt.sum(1, keepdims=True)\n        L = np.log((cnt + self.alpha) / (tot + self.alpha * V)).T\n        L = L - L.mean(1, keepdims=True)\n        self.L_ = np.clip(L, -self.clip, self.clip)\n        return self\n\n    def contribution(self, texts):\n        n_docs = len(texts)\n        if len(self.vocab_) == 0:\n            return np.zeros((n_docs, self.n_classes_))\n        X = self._matrix(self._docs(texts))\n        n = np.asarray(X.sum(1)).ravel()\n        return np.asarray(X @ self.L_) / np.sqrt(np.maximum(n, 1.0))[:, None]\n\n\ndef _finalize(p):\n    p = np.clip(p, PROB_FLOOR, 1.0 - PROB_FLOOR)\n    return p / p.sum(1, keepdims=True)\n\n\ndef prob_to_logit(p1):\n    p1 = np.clip(np.asarray(p1, dtype=float), PROB_FLOOR, 1.0 - PROB_FLOOR)\n    return np.log(p1 / (1.0 - p1))\n\n\ndef clipped_logit(p1, clip=LOGIT_CLIP):\n    return np.clip(prob_to_logit(p1), -float(clip), float(clip))\n\n\ndef apply_temperature(P, T):\n    logit = np.clip(prob_to_logit(P[:, 1]) / float(T), -50.0, 50.0)\n    p1 = 1.0 / (1.0 + np.exp(-logit))\n    return _finalize(np.column_stack([1.0 - p1, p1]))\n\n\ndef fit_temperature(P, y):\n    from sklearn.metrics import log_loss\n    res = minimize_scalar(lambda t: log_loss(y, apply_temperature(P, t), labels=[0, 1]),\n                          bounds=(0.2, 10.0), method="bounded")\n    return float(res.x)\n\n\ndef binary_stats_at_preds(y, pred):\n    y = np.asarray(y, dtype=int); pred = np.asarray(pred, dtype=int)\n    tp = float(((y == 1) & (pred == 1)).sum())\n    tn = float(((y == 0) & (pred == 0)).sum())\n    fp = float(((y == 0) & (pred == 1)).sum())\n    fn = float(((y == 1) & (pred == 0)).sum())\n    prec1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n    rec1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n    f1_1 = (2 * prec1 * rec1 / (prec1 + rec1)) if (prec1 + rec1) > 0 else 0.0\n    prec0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0\n    rec0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0\n    f1_0 = (2 * prec0 * rec0 / (prec0 + rec0)) if (prec0 + rec0) > 0 else 0.0\n    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0\n    return dict(f1_1=f1_1, f1_0=f1_0, macro_f1=0.5 * (f1_0 + f1_1), prec=prec1, rec=rec1, fpr=fpr)\n\n\ndef threshold_table(P, y, grid=np.linspace(-2.5, 2.5, 101)):\n    logit = prob_to_logit(P[:, 1])\n    y = np.asarray(y, dtype=int)\n    f1s, mf1s, prs, rcs, fprs = [], [], [], [], []\n    for t in grid:\n        st = binary_stats_at_preds(y, (logit >= t).astype(int))\n        f1s.append(st["f1_1"]); mf1s.append(st["macro_f1"])\n        prs.append(st["prec"]); rcs.append(st["rec"]); fprs.append(st["fpr"])\n    return dict(thresholds=np.asarray(grid, float), f1=np.asarray(f1s), macro_f1=np.asarray(mf1s),\n                precision=np.asarray(prs), recall=np.asarray(rcs), fpr=np.asarray(fprs))\n\n\ndef pick_threshold(tab, mode="f1_opt", min_precision=0.70):\n    ts = tab["thresholds"]\n    mag = np.abs(ts)\n    if mode == "argmax":\n        return 0.0, dict(mode_used="argmax", feasible=True, threshold_prob=0.5,\n                         f1=float("nan"), macro_f1=float("nan"),\n                         precision=float("nan"), recall=float("nan"), fpr=float("nan"))\n    n = len(ts)\n    i_f1 = max(range(n), key=lambda i: (round(tab["f1"][i], 9), -mag[i]))\n    i_mf1 = max(range(n), key=lambda i: (round(tab["macro_f1"][i], 9), -mag[i]))\n    chosen, used, feas = (i_mf1 if mode == "macro_f1" else i_f1), ("macro_f1" if mode == "macro_f1" else "f1_opt"), True\n    if mode == "recall_at_precision":\n        ok = [i for i in range(n) if tab["precision"][i] >= min_precision]\n        if ok:\n            chosen = max(ok, key=lambda i: (round(tab["recall"][i], 9), round(tab["f1"][i], 9), -mag[i]))\n            used = "recall_at_precision"\n        else:\n            feas = False\n    t_star = float(ts[chosen])\n    p_star = float(1.0 / (1.0 + np.exp(-t_star)))\n    return t_star, dict(mode_used=used, feasible=feas, threshold_prob=p_star,\n                        f1=float(tab["f1"][chosen]), macro_f1=float(tab["macro_f1"][chosen]),\n                        precision=float(tab["precision"][chosen]), recall=float(tab["recall"][chosen]),\n                        fpr=float(tab["fpr"][chosen]))\n\n\ndef tune_threshold(P, y, grid=np.linspace(-2.5, 2.5, 101), mode="f1_opt", min_precision=0.70):\n    t, info = pick_threshold(threshold_table(P, y, grid), mode, min_precision)\n    return t, info["f1"]\n\n\nclass HeaderAgent:\n    def __init__(self, features=None, n_bins=24, alpha=1.0, lik_scale=1.0,\n                 max_missing_log_odds=1.5, feat_bound=2.5, use_missingness_features=True,\n                 group_weighting="flat", subject_tokens=False, token_weight=1.0, smoothing="marginal"):\n        raw = list(features) if features is not None else ALL_FEATURES\n        self.use_missingness_features = bool(use_missingness_features)\n        self.features = raw if self.use_missingness_features else [f for f in raw if f not in MISSINGNESS_FEATURES]\n        self.n_bins, self.alpha, self.lik_scale = int(n_bins), float(alpha), float(lik_scale)\n        self.max_missing_log_odds, self.feat_bound = float(max_missing_log_odds), float(feat_bound)\n        self.group_weighting, self.subject_tokens = group_weighting, bool(subject_tokens)\n        self.token_weight, self.smoothing = float(token_weight), smoothing\n        self.temperature = 1.0\n        self.threshold_logit = 0.0\n        self.classes_ = np.array([0, 1])\n        self.is_fitted_ = False\n\n    def _weights(self):\n        if self.group_weighting != "inv_group":\n            return np.ones(len(self.features))\n        sizes = {g: sum(f in self.features for f in fs) for g, fs in FEATURE_GROUPS.items()}\n        mean_n = np.mean([n for n in sizes.values() if n > 0])\n        w = []\n        for f in self.features:\n            g = next(g for g, fs in FEATURE_GROUPS.items() if f in fs)\n            w.append(np.sqrt(mean_n / sizes[g]))\n        return np.array(w)\n\n    def fit_features(self, Xf, y):\n        y = np.asarray(y, dtype=int)\n        self.preprocessor_ = HeaderPreprocessor(self.features, self.n_bins).fit(Xf)\n        Z = self.preprocessor_.transform(Xf)\n        raw = np.bincount(y, minlength=N_CLASSES).astype(float)\n        self.training_prior_ = raw[:N_CLASSES] / raw.sum()\n        self.model_ = SmoothedCategoricalNB(\n            alpha=self.alpha, lik_scale=self.lik_scale,\n            max_missing_log_odds=self.max_missing_log_odds, feat_bound=self.feat_bound,\n            smoothing=self.smoothing).fit(Z, y, self.preprocessor_.n_categories_, self.preprocessor_, self._weights())\n        self.token_nb_ = None\n        if self.subject_tokens:\n            self.token_nb_ = SubjectTokenNB().fit(Xf["u_norm"].tolist(), y, N_CLASSES)\n        self.temperature, self.threshold_logit = 1.0, 0.0\n        self.is_fitted_ = True\n        return self\n\n    def set_temperature(self, T):\n        self.temperature = float(T)\n\n    def set_threshold_logit(self, t):\n        self.threshold_logit = float(t)\n\n    def proba_features(self, Xf, calibrated=True):\n        if not self.is_fitted_:\n            raise RuntimeError("Model is not fitted.")\n        lj = self.model_.log_joint(self.preprocessor_.transform(Xf))\n        if self.token_nb_ is not None:\n            lj = lj + self.token_weight * self.token_nb_.contribution(Xf["u_norm"].tolist())\n        lj = lj - lj.max(1, keepdims=True)\n        p = np.exp(lj)\n        p = _finalize(p / p.sum(1, keepdims=True))\n        return apply_temperature(p, self.temperature) if calibrated else p\n\n    def predict_label_features(self, Xf):\n        P = self.proba_features(Xf, calibrated=True)\n        return (prob_to_logit(P[:, 1]) >= self.threshold_logit).astype(int)\n\n    @staticmethod\n    def completeness(Xf):\n        s = Xf["s_missing"].to_numpy().astype(int)\n        r = Xf["r_missing"].to_numpy().astype(int)\n        d = (Xf["d_date_status"] == 0).to_numpy().astype(int)\n        u = Xf["u_missing"].to_numpy().astype(int)\n        return ((1 - s) + (1 - r) + (1 - d) + (1 - u)) / 4.0\n\n    def detailed_features(self, Xf, index=None):\n        P = self.proba_features(Xf, calibrated=True)\n        comp = self.completeness(Xf)\n        nb_score = P[:, 1]\n        return pd.DataFrame({\n            "p_safe": P[:, 0],\n            "nb_score": nb_score,\n            "nb_logit": clipped_logit(nb_score),\n            "pred_class": (prob_to_logit(nb_score) >= self.threshold_logit).astype(int),\n            "header_completeness": comp,\n            "insufficient_evidence": (comp <= 0.5).astype(int)},\n            index=Xf.index if index is None else index)\n\n    def fit(self, X_emails, y):\n        return self.fit_features(extract_features(_ensure_df(X_emails)), y)\n\n    def predict_proba(self, X_emails, calibrated=True):\n        df = _ensure_df(X_emails)\n        return pd.DataFrame(self.proba_features(extract_features(df), calibrated), index=df.index, columns=list(CLASS_NAMES))\n\n    def predict_detailed(self, X_emails):\n        df = _ensure_df(X_emails)\n        return self.detailed_features(extract_features(df), df.index)\n\n    def save(self, path):\n        import joblib\n        joblib.dump({"agent": self, "version": __version__}, path)\n\n    @classmethod\n    def load(cls, path):\n        import joblib\n        return joblib.load(path)["agent"]\n\n\ndef score_emails(agent, emails):\n    p = agent.predict_proba(emails, calibrated=True).to_numpy(dtype=float)\n    return np.clip(p[:, 1], 0.0, 1.0)\n\n\ndef score_email(agent, email):\n    return float(score_emails(agent, [email])[0])\n'

import importlib
MODULE_PATH = OUT / 'agent1_header.py'
MODULE_PATH.write_text(HEADER_SRC, encoding='utf-8')
sys.path.insert(0, str(OUT.resolve()))
os.environ['PYTHONPATH'] = str(OUT.resolve()) + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.modules.pop('agent1_header', None)
import agent1_header as A1
CLASS_NAMES = list(A1.CLASS_NAMES)
assert CLASS_NAMES == ['safe', 'not_safe']
print('Agent 1 module', A1.__version__, '| class order', CLASS_NAMES)

Agent 1 module 8.0.0-binary | class order ['safe', 'not_safe']


## 2. Binary Unit Testing and Parser Robustness Verification

Verifies parser invariants (B2–B5, Q1–Q2), binary $K=2$ likelihood normalization (N1) and `sklearn.CategoricalNB` parity, subject-token NB polarity, temperature scaling on binary log-odds, threshold picking (`argmax`, `f1_opt`, `recall_at_precision`), hostile inputs, and single-vs-batch parity.

In [7]:
import numpy as np, pandas as pd
from sklearn.naive_bayes import CategoricalNB


def run_unit_tests(A):
    ok = []
    def check(name, cond):
        assert cond, f"FAILED: {name}"
        ok.append(name)

    base = {"sender": "a@b.com", "receiver": "x@y.com", "date": "Mon, 12 Oct 2026 14:00:00 +0000", "subject": "hi"}

    F = A.extract_features(pd.DataFrame([
        {**base}, {**base, "date": "Fri, 16 Oct 2026 10:00:00 -0500"}, {**base, "date": "Sun, 18 Oct 2026 23:00:00 GMT"}]))
    check("B2 weekday computed (0,4,6)", F["d_weekday"].tolist() == [0, 4, 6])
    check("B2 tz parsed (0,-5,0)", F["d_tz_h"].tolist() == [0, -5, 0])
    F = A.extract_features(pd.DataFrame([{**base, "date": "Thu, 30 Feb 2026 10:00:00 +0000"}]))
    check("B3 impossible calendar date -> status 3", F["d_date_status"].iloc[0] == 3)

    rng = np.random.default_rng(0)
    N = 400
    rows = [{**base, "sender": f"u{i}@d{i%9}.com",
             "date": ["Mon, 12 Oct 2026 14:00:00 +0000", "garbage", "", "Tue, 13 Oct 2026 10:00:00"][i % 4]} for i in range(N)]
    df = pd.DataFrame(rows); y = rng.integers(0, 2, N)
    ag = A.HeaderAgent(n_bins=8).fit(df, y)
    Z = ag.preprocessor_.transform(A.extract_features(df.iloc[:4]))
    names, K = ag.preprocessor_.features, ag.preprocessor_.n_categories_
    ab = lambda f, r: Z[r, names.index(f)] == K[names.index(f)] - 1
    check("B3 garbage date: hour/weekday/tz abstain", ab("d_hour", 1) and ab("d_weekday", 1) and ab("d_tz_h", 1))
    check("B3 missing date: hour/weekday/tz abstain", ab("d_hour", 2) and ab("d_weekday", 2) and ab("d_tz_h", 2))
    check("B3 date without tz: hour kept, tz abstains", (not ab("d_hour", 3)) and ab("d_tz_h", 3))
    check("B3 valid date: nothing abstains", not (ab("d_hour", 0) or ab("d_weekday", 0) or ab("d_tz_h", 0)))

    d4 = pd.DataFrame([{**base, "receiver": "undisclosed-recipients:;"}, {**base, "receiver": None}, {**base}])
    F4 = A.extract_features(d4)
    check("B4 undisclosed != missing", F4["r_missing"].tolist() == [0, 1, 0] and F4["r_has_undisclosed"].tolist() == [1, 0, 0])
    Z4 = ag.preprocessor_.transform(F4)
    j = names.index("r_has_undisclosed")
    check("B4 undisclosed feature is NOT abstained", Z4[0, j] == 1 and Z4[1, j] == K[j] - 1)

    F4s = A.extract_features(pd.DataFrame([{**base, "receiver": r} for r in
        [None, "undisclosed-recipients:;", "x@y.com", "a@b.com, c@d.com", "Undisclosed <u@b.com>"]]))
    check("R1 r_state = missing/undisclosed/single/several/undisclosed+addr", F4s["r_state"].tolist() == [0, 1, 2, 3, 1])

    doms = {"us.com": 0, "its.com": 0, "up.org": 0, "city.com": 0, "apply.com": 0, "dell.com": 0,
            "paypa1.com": 1, "paypall.com": 1, "goggle.com": 1, "paypal.com": 0, "ups.com": 0, "irs.gov": 0}
    F5 = A.extract_features(pd.DataFrame([{**base, "sender": f"a@{d}"} for d in doms]))
    check("B5 typo-brand", F5["s_typo_brand"].tolist() == list(doms.values()))

    # Binary K=2 likelihood parity & N1
    rng = np.random.default_rng(3)
    n = 50000; yy = rng.choice(2, n, p=[.45, .55])
    X = np.column_stack([(rng.random(n) < np.array([.2, .8])[yy]).astype(int),
                         rng.integers(0, 5, n), rng.integers(0, 4, n)])
    Kc = [3 + 1, 5 + 1, 4 + 1]
    ours = A.SmoothedCategoricalNB(alpha=1.0, feat_bound=0, max_missing_log_odds=0, smoothing="laplace").fit(X, yy, Kc)
    sk = CategoricalNB(alpha=1.0, min_categories=[3, 5, 4]).fit(X, yy)
    check("K=2 parity with sklearn.CategoricalNB", np.allclose(ours.predict_proba(X), sk.predict_proba(X), atol=1e-9))

    Xa = X.copy(); Ka = [4, 6, 5]
    mask = rng.random(n) < np.array([.05, .7])[yy]
    Xa[mask, 1] = Ka[1] - 1
    m = A.SmoothedCategoricalNB(alpha=1.0, feat_bound=0, max_missing_log_odds=0).fit(Xa, yy, Ka)
    P = np.exp(m.tables_[1][: Ka[1] - 1]).T
    check("N1 P(cat|class, not abstained) sums to 1 per class (K=2)", np.allclose(P.sum(1), 1.0))
    spread = np.abs(m.tables_[1][: Ka[1] - 1] - m.tables_[1][: Ka[1] - 1].mean(1, keepdims=True)).max()
    check("N1 class-dependent abstain rate does not leak", spread < 0.15)
    check("abstain code contributes 0", np.all(m.tables_[1][Ka[1] - 1] == 0))

    # Subject-token block (K=2)
    tx = ["verify your account now"] * 60 + ["team meeting notes"] * 60
    ty = np.array([1] * 60 + [0] * 60)
    tk = A.SubjectTokenNB(min_df=5).fit(tx, ty, 2)
    c = tk.contribution(["verify account", "zzz qqq", "", None])
    check("token block K=2: malicious words push class 1, unknown/empty/None=0",
          c[0].argmax() == 1 and np.allclose(c[1:], 0))

    # Hostile inputs & K=2 output shape
    odd = [{}, {"sender": None}, {"sender": float("nan"), "subject": 5, "date": 1.5, "receiver": b"x@y.com"},
           {"sender": "no-at-sign", "receiver": "<>", "date": "\x00\x01", "subject": "\u202e" * 50},
           {"sender": "=?utf-8?b?4pyTIFRlc3Q=?= <A@B.CO.UK>", "receiver": "a@b.com, c@d.com", "subject": "RE: RE: Fwd: x" * 100}]
    ag3 = A.HeaderAgent(n_bins=8, subject_tokens=True).fit(df.assign(subject="verify your account"), y)
    p = ag3.predict_proba(odd).to_numpy()
    check("hostile inputs: shape (n,2), finite, in [0,1], rows sum to 1",
          p.shape == (len(odd), 2) and np.isfinite(p).all() and (p >= 0).all() and np.allclose(p.sum(1), 1.0))
    check("class order binary", A.CLASS_NAMES == ("safe", "not_safe") and list(ag3.predict_proba(odd).columns) == list(A.CLASS_NAMES))
    check("single == batch scoring", abs(A.score_email(ag3, odd[2]) - A.score_emails(ag3, odd)[2]) < 1e-12)

    # Temperature scaling & binary threshold picker
    P0 = ag3.proba_features(A.extract_features(df), calibrated=False)
    T = A.fit_temperature(P0, y)
    Pc = A.apply_temperature(P0, T)
    check("binary temperature keeps rows normalised", np.allclose(Pc.sum(1), 1.0) and 0.2 <= T <= 10)
    rr = np.random.default_rng(5); ys = rr.choice(2, 3000, p=[.45, .55])
    lg_s = rr.normal(size=(3000, 2)) + 1.2 * np.eye(2)[ys]; Ps = np.exp(lg_s - lg_s.max(1, keepdims=True)); Ps /= Ps.sum(1, keepdims=True)
    tab = A.threshold_table(Ps, ys, grid=np.linspace(-2, 2, 41))
    t_a, _ = A.pick_threshold(tab, "argmax")
    check("argmax threshold is 0.0", t_a == 0.0)
    t_x, i_x = A.pick_threshold(tab, "recall_at_precision", min_precision=1.01)
    check("infeasible precision floor falls back to f1_opt", (not i_x["feasible"]) and i_x["mode_used"] == "f1_opt")
    t_y, i_y = A.pick_threshold(tab, "recall_at_precision", min_precision=0.70)
    check("feasible precision floor honoured", i_y["feasible"] and i_y["precision"] >= 0.70)

    det_b = ag3.predict_detailed(odd); det_1 = pd.concat([ag3.predict_detailed([o]) for o in odd], ignore_index=True)
    check("single == batch for predict_detailed", np.allclose(det_b.reset_index(drop=True).to_numpy(float), det_1.to_numpy(float)))
    return ok


passed = run_unit_tests(A1)
print(f"{len(passed)} binary unit checks passed")

24 binary unit checks passed


## 3. Dataset Loading and Binary Target Formulation

Uses the dataset's raw binary `label` directly (`0 = Safe`, `1 = Not Safe`). The `source` column is retained strictly as a **diagnostic** (never as a predictive feature) to verify that `Nazario` and `Nigerian_Fraud` contain only class `1` and to run LOSO cross-corpus diagnostics.

In [8]:
CSV_PATH = None
searched = [
    CSV_PATH,
    os.environ.get("EMAILS_CSV", ""),
    "data/processed/emails_clean.csv",
    "emails_clean.csv",
]
searched += glob.glob("/kaggle/input/**/emails_clean.csv", recursive=True) + glob.glob("**/emails_clean.csv", recursive=True)
DATA_PATH = next((os.path.abspath(c) for c in searched if c and os.path.isfile(c)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "emails_clean.csv was not found.\n  -> Kaggle: add the dataset that contains it to the notebook (Add Input).\n"
        "  -> Local : place it at data/processed/emails_clean.csv or next to this notebook, or set CSV_PATH / EMAILS_CSV.\n"
        "Nothing was trained and no data was generated.")
print("Dataset:", DATA_PATH)
df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[""])
required = ["email_id", "split", "source", "sender", "receiver", "date", "subject", "label"]
miss = [c for c in required if c not in df.columns]
if miss: raise ValueError(f"missing columns: {miss} (found: {list(df.columns)})")
if df["email_id"].isna().any() or not df["email_id"].is_unique: raise ValueError("email_id must be unique and non-null")
if df["label"].isna().any() or df["source"].isna().any(): raise ValueError("label and source must not be empty")
aliases = {"train": "train", "training": "train", "val": "val", "validation": "val", "valid": "val", "test": "test", "testing": "test"}
df["split"] = df["split"].astype(str).str.strip().str.lower().map(aliases)
if df["split"].isna().any() or set(df["split"]) != {"train", "val", "test"}:
    raise ValueError("split must contain exactly train / val / test")
train_df, val_df, test_df = [df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")]

norm_source = lambda x: "".join(ch for ch in str(x).lower() if ch.isalnum())

def get_binary_target(frame):
    num = pd.to_numeric(frame["label"], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(num).all() or not np.all(np.isin(num, [0.0, 1.0])):
        bad = np.unique(num[~np.isfinite(num) | ~np.isin(num, [0.0, 1.0])])[:10].tolist()
        raise ValueError(f"unexpected binary labels {bad} (expected exact 0 or 1 without fractional/NaN values)")
    return num.astype(int)

y_train = get_binary_target(train_df)
src_train = train_df["source"].map(norm_source).to_numpy()
if set(np.unique(y_train)) != {0, 1}:
    raise ValueError("training split lacks a binary class")

# Diagnostic-only inspection across ALL splits (never passed to fitting, CV, calibration, or threshold tuning)
_y_diag = get_binary_target(df)
_src_diag = df["source"].map(norm_source).to_numpy()
for ph_src in ("nazario", "nigerianfraud"):
    m = _src_diag == ph_src
    if m.any():
        assert set(np.unique(_y_diag[m])) == {1}, f"{ph_src} unexpectedly contains class 0 across splits!"

print("Rows:", dict(train=len(train_df), val=len(val_df), test=len(test_df)))
print("Training class counts [safe, not_safe]:", np.bincount(y_train).tolist(), "| prior", (np.bincount(y_train) / len(y_train)).round(3).tolist())
print("\nDIAGNOSTIC ONLY (labels never used for val/test tuning): source x (split, y_binary):")
print(pd.crosstab(df["source"], [df["split"], _y_diag], rownames=["source"], colnames=["split", "y_binary"]))
for sp in ("train", "val", "test"):
    msp = df["split"].to_numpy() == sp
    cnt = np.bincount(_y_diag[msp], minlength=2)
    print(f"  diagnostic {sp:5s}: counts={cnt.tolist()} | P(not_safe)={cnt[1]/cnt.sum():.4f}")
del _y_diag, _src_diag

Dataset: /kaggle/input/datasets/itsduaa/cleanone/emails_clean.csv
Rows: {'train': 34256, 'val': 7340, 'test': 7343}
Training class counts [safe, not_safe]: [14952, 19304] | prior [0.436, 0.564]

DIAGNOSTIC ONLY (labels never used for val/test tuning): source x (split, y_binary):
split           test        train          val      
y_binary           0     1      0      1     0     1
source                                              
CEAS_08         2543  3111  12136  14813  2592  3164
Nazario            0   236      0   1062     0   228
Nigerian_Fraud     0   525      0   2255     0   490
SpamAssasin      662   266   2816   1174   612   254
  diagnostic train: counts=[14952, 19304] | P(not_safe)=0.5635
  diagnostic val  : counts=[3204, 4136] | P(not_safe)=0.5635
  diagnostic test : counts=[3205, 4138] | P(not_safe)=0.5635


## 4. Feature Extraction and Cross-Split Overlap Diagnostics
Extracts the 36 header features and quantifies sender-identity and subject-template overlap between `train` and `val`/`test`.

In [9]:
X_train = A1.extract_features(train_df[HEADER_COLS])
X_val   = A1.extract_features(val_df[HEADER_COLS])
X_test  = A1.extract_features(test_df[HEADER_COLS])
print("model features:", len(A1.ALL_FEATURES))

def group_keys(Xf, ids, level):
    out = []
    for addr, dom, i in zip(Xf["s_addr"], Xf["s_dom"], ids):
        if level == "identity": g = A1.canonical_sender_identity(dom, addr)
        else: g = f"addr:{addr}" if addr else None
        out.append(g if g else f"missing:{i}")
    return np.asarray(out, dtype=object)

G_ID   = group_keys(X_train, train_df["email_id"], "identity")
G_ADDR = group_keys(X_train, train_df["email_id"], "address")
templates = X_train["u_norm"].to_numpy()
vc = pd.Series(G_ID).value_counts()
print(f"\nSender groups (identity level): {len(vc):,} | largest group = {vc.iloc[0]:,} rows ({vc.iloc[0]/len(G_ID):.1%}) -> {vc.index[0]}")

def keysets(Xf, ids):
    return dict(addr=set(group_keys(Xf, ids, "address")), ident=set(group_keys(Xf, ids, "identity")),
                tpl={t for t in Xf["u_norm"] if t}, dup=set(zip(Xf["s_addr"], Xf["u_norm"])))
tr_k = keysets(X_train, train_df["email_id"]); rows = []
for name, Xf, fr in (("val", X_val, val_df), ("test", X_test, test_df)):
    a, i = group_keys(Xf, fr["email_id"], "address"), group_keys(Xf, fr["email_id"], "identity")
    rows.append({"split": name,
        "sender_address_seen_in_train": np.mean([k in tr_k["addr"] for k in a]),
        "sender_identity_seen_in_train": np.mean([k in tr_k["ident"] for k in i]),
        "subject_template_seen": np.mean([t in tr_k["tpl"] for t in Xf["u_norm"] if t]),
        "near_duplicate_sender+subject_seen": np.mean([k in tr_k["dup"] for k in zip(Xf["s_addr"], Xf["u_norm"])])})
print("\nShare of eval rows whose key also occurs in TRAIN:")
print(pd.DataFrame(rows).round(3).to_string(index=False))

dk = pd.Series(list(zip(X_train["s_addr"], X_train["u_norm"])))
dupmask = dk.duplicated(keep=False).to_numpy() & (X_train["s_addr"].to_numpy() != "")
conf = pd.DataFrame({"k": dk, "y": y_train}).groupby("k")["y"].nunique()
print(f"\nTrain rows sharing (sender, subject-template) with another row: {dupmask.mean():.1%}; keys with conflicting labels: {(conf > 1).sum()}")
rawdup = train_df.duplicated(HEADER_COLS, keep=False)
print(f"Train rows whose four raw header strings are identical to another row: {rawdup.mean():.1%}")

print("\nMissing-sender rows by binary class:")
print(pd.crosstab(X_train.s_missing.map({0: "sender present", 1: "sender missing"}), pd.Series(y_train).map(dict(enumerate(CLASS_NAMES)))).to_string())

model features: 36

Sender groups (identity level): 15,865 | largest group = 463 rows (1.4%) -> dom:apache.org

Share of eval rows whose key also occurs in TRAIN:
split  sender_address_seen_in_train  sender_identity_seen_in_train  subject_template_seen  near_duplicate_sender+subject_seen
  val                         0.428                          0.637                  0.670                               0.207
 test                         0.436                          0.637                  0.663                               0.204

Train rows sharing (sender, subject-template) with another row: 18.9%; keys with conflicting labels: 2
Train rows whose four raw header strings are identical to another row: 0.1%

Missing-sender rows by binary class:
col_0           not_safe   safe
s_missing                      
sender missing       632      0
sender present     18672  14952


## 5. Grouped Cross-Validation, Candidate Grid (96 Configs), and Binary Metrics

Constructs 5-fold `StratifiedGroupKFold` grouped by canonical sender identity, builds the **96-candidate** binary grid (`4 feature_sets × 3 lik_scale × 2 group_weighting × 2 subject_tokens × 2 alpha`), and defines the binary evaluation suite:
`accuracy`, `precision`, `recall`, `f1`, `macro_f1`, `roc_auc`, `ap`, `fpr`, `fnr`, `log_loss`, `ece`, and 2×2 confusion matrix.

In [10]:
n_train = len(y_train)
ALL_IDX = np.arange(n_train)
assert len(X_train) == n_train == len(train_df) == len(G_ID) == len(templates)
assert (X_train.index.to_numpy() == ALL_IDX).all()

def _tpl_set(ix):
    return {t for t in templates[ix] if t}

def make_group_folds(idx, groups, n_splits, seed, purge=False):
    idx = np.asarray(idx)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for k, (tr_rel, va_rel) in enumerate(cv.split(np.zeros(len(idx)), y_train[idx], groups=groups[idx])):
        tr, va = idx[tr_rel], idx[va_rel]
        if set(groups[tr]) & set(groups[va]): raise AssertionError(f"group leakage in fold {k}")
        if purge:
            bad = _tpl_set(va)
            tr = tr[np.array([(not t) or (t not in bad) for t in templates[tr]], dtype=bool)]
            if _tpl_set(tr) & _tpl_set(va): raise AssertionError(f"template leakage in fold {k}")
        if len(np.unique(y_train[tr])) != 2: raise ValueError(f"fold {k}: training part lacks a binary class")
        if len(va) == 0: raise ValueError(f"fold {k}: empty evaluation part")
        out.append((tr, va))
    cov = np.sort(np.concatenate([va for _, va in out]))
    if not np.array_equal(cov, np.sort(idx)): raise AssertionError("evaluation parts must partition idx exactly once")
    return out

try:
    GROUPS, GROUP_LEVEL = G_ID, "identity (domain / free-mail address)"
    folds = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED)
except ValueError as e:
    print("!! identity-level grouping unusable:", e, "-> falling back to address-level groups")
    GROUPS, GROUP_LEVEL = G_ADDR, "address"
    folds = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED)
print("CV grouping:", GROUP_LEVEL)
FOLD_ID = np.full(n_train, -1)
for k, (tr, va) in enumerate(folds):
    FOLD_ID[va] = k
    tr_t, v = _tpl_set(tr), [t for t in templates[va] if t]
    print(f"fold {k}: train={len(tr):,} holdout={len(va):,} classes={np.bincount(y_train[va], minlength=2).tolist()} seen-template={np.mean([t in tr_t for t in v]):.1%}")
assert (FOLD_ID >= 0).all()

_nc = lambda fl: [f for f in fl if f not in A1.CLOCK_FEATURES]
FEATURE_SETS = {"full": list(A1.ALL_FEATURES), "no_clock": _nc(A1.ALL_FEATURES),
                "compact_recv": list(A1.ALL_FEATURES_COMPACT), "compact_no_clock": _nc(A1.ALL_FEATURES_COMPACT)}

def build_grid(kind):
    if kind == "full":
        lik = (1.0, 2.0) if QUICK_SEARCH else (1.0, 1.5, 2.0)
        gw  = ("flat",) if QUICK_SEARCH else ("flat", "inv_group")
        tok = (False, True)
        alp = (1.0,) if QUICK_SEARCH else (0.5, 1.0)
    elif kind == "reduced":
        lik, gw, tok, alp = (1.0,), (("flat",) if QUICK_SEARCH else ("flat", "inv_group")), (False, True), ((1.0,) if QUICK_SEARCH else (0.5, 1.0))
    elif kind == "stress":
        lik, gw, tok, alp = (1.0, 1.5, 2.0), ("flat",), (False, True), (1.0,)
    else:
        raise ValueError(kind)
    return [dict(feature_set=fs, n_bins=16, alpha=a, lik_scale=l, group_weighting=g, subject_tokens=t,
                 feat_bound=2.5, max_missing_log_odds=1.0)
            for fs in FEATURE_SETS for l in lik for g in gw for t in tok for a in alp]

FULL_GRID, NESTED_GRID, STRESS_GRID = build_grid("full"), build_grid(NESTED_GRID_MODE), build_grid("stress")
REDUCED_GRID = build_grid("reduced")
if not QUICK_SEARCH:
    assert NESTED_GRID_MODE == "full" and len(NESTED_GRID) == 96 == len(FULL_GRID)
print(f"candidate grids: full={len(FULL_GRID)} nested({NESTED_GRID_MODE})={len(NESTED_GRID)} reduced={len(REDUCED_GRID)} stress={len(STRESS_GRID)}")
complexity = lambda c: len(FEATURE_SETS[c["feature_set"]]) // 8 + c["subject_tokens"] * 2 + (c["group_weighting"] != "flat")

def cfg_to_agent_kwargs(c):
    kw = {k: v for k, v in c.items() if k != "feature_set"}
    kw["features"] = FEATURE_SETS[c["feature_set"]]
    return kw

_UNUSED = ["s_addr", "s_dom"]
def _slim(Xf, cfg):
    drop = _UNUSED + ([] if cfg["subject_tokens"] else ["u_norm"])
    return Xf.drop(columns=[c for c in drop if c in Xf.columns])

def _fit_proba(kw, Xtr, ytr, Xva):
    import agent1_header as A
    m = A.HeaderAgent(**kw).fit_features(Xtr, ytr)
    return m.proba_features(Xva, calibrated=False)

def run_tasks(tasks):
    if N_JOBS <= 1 or len(tasks) < 2: return [_fit_proba(*t) for t in tasks]
    return Parallel(n_jobs=N_JOBS, backend="loky")(delayed(_fit_proba)(*t) for t in tasks)

def predict_from_P(P, t_logit=0.0):
    return (A1.prob_to_logit(P[:, 1]) >= float(t_logit)).astype(int)

def safe_ap(ytrue_bool, score):
    yb = np.asarray(ytrue_bool, dtype=int)
    return float(average_precision_score(yb, score)) if 0 < yb.sum() < len(yb) else float("nan")

def safe_roc_auc(ytrue_bool, score):
    yb = np.asarray(ytrue_bool, dtype=int)
    return float(roc_auc_score(yb, score)) if 0 < yb.sum() < len(yb) else float("nan")

def ece_binary(P, y, bins=10):
    p1 = P[:, 1]; y = np.asarray(y, dtype=float)
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p1 >= lo) & (p1 < hi) if hi < 1.0 else (p1 >= lo) & (p1 <= hi)
        if m.any(): e += m.mean() * abs(p1[m].mean() - y[m].mean())
    return float(e)

def metrics(y, P, t_logit=0.0, pred=None):
    y = np.asarray(y, dtype=int)
    pred = predict_from_P(P, t_logit) if pred is None else np.asarray(pred, dtype=int)
    both = len(np.unique(y)) == 2
    cm = confusion_matrix(y, pred, labels=[0, 1])
    tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    present = [c for c in (0, 1) if sup[c] > 0]
    return dict(
        n=len(y), n_safe=int(sup[0]), n_not_safe=int(sup[1]),
        accuracy=float(accuracy_score(y, pred)),
        precision=float(pr[1]) if (both and (tp + fp) > 0) else float("nan"),
        recall=float(rc[1]) if sup[1] > 0 else float("nan"),
        f1=float(f1[1]) if (both and sup[1] > 0) else float("nan"),
        macro_f1=float(np.mean(f1[present])) if both else float("nan"),
        roc_auc=safe_roc_auc(y == 1, P[:, 1]),
        ap=safe_ap(y == 1, P[:, 1]),
        fpr=float(fp / (fp + tn)) if (fp + tn) > 0 else float("nan"),
        fnr=float(fn / (fn + tp)) if (fn + tp) > 0 else float("nan"),
        log_loss=float(log_loss(y, P, labels=[0, 1])) if both else float("nan"),
        ece=ece_binary(P, y),
        tn=tn, fp=fp, fn=fn, tp=tp)

def full_report(y, P, t_logit, title, pred=None):
    pred = predict_from_P(P, t_logit) if pred is None else pred
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    print(f"\n=== {title} ===")
    print(pd.DataFrame({"precision": pr, "recall": rc, "f1": f1, "support": sup}, index=CLASS_NAMES).round(4).to_string())
    print("confusion (rows=true, cols=pred):")
    print(pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]), index=[f"true_{c}" for c in CLASS_NAMES], columns=[f"pred_{c}" for c in CLASS_NAMES]).to_string())
    m = metrics(y, P, t_logit, pred)
    print({k: round(float(v), 4) for k, v in m.items() if k not in ("n", "n_safe", "n_not_safe", "tn", "fp", "fn", "tp") and not np.isnan(v)})

def by_source(y, P, pred, sources):
    rows = []
    for s in np.unique(sources):
        m = sources == s
        r = dict(source=s, **metrics(y[m], P[m], pred=pred[m]))
        rows.append(r)
    return pd.DataFrame(rows)

def decision_name(mode=None, floor=None):
    mode = DECISION_MODE if mode is None else mode
    return {"argmax": "argmax", "f1_opt": "f1_opt", "macro_f1": "macro_f1"}.get(
        mode, f"recall@P>={(MIN_NOTSAFE_PRECISION if floor is None else floor):.2f}")

print("binary helpers ready | decision rule:", decision_name())

CV grouping: identity (domain / free-mail address)
fold 0: train=27,492 holdout=6,764 classes=[2829, 3935] seen-template=59.3%
fold 1: train=27,777 holdout=6,479 classes=[2537, 3942] seen-template=60.1%
fold 2: train=27,346 holdout=6,910 classes=[3140, 3770] seen-template=55.9%
fold 3: train=27,111 holdout=7,145 classes=[3534, 3611] seen-template=56.2%
fold 4: train=27,298 holdout=6,958 classes=[2912, 4046] seen-template=60.3%
candidate grids: full=96 nested(full)=96 reduced=32 stress=24
binary helpers ready | decision rule: f1_opt


## 6. Diagnostic: Feature-Group Source Identifiability Audit
Measures how strongly each header feature group distinguishes the 4 corpora (class-agnostic).

In [11]:
from sklearn.naive_bayes import CategoricalNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import balanced_accuracy_score

src_codes, src_names = pd.factorize(src_train)
GROUP_DEFS = {
    "sender (all s_*)": A1.FEATURE_GROUPS["sender"], "receiver (all r_*)": A1.FEATURE_GROUPS["receiver"],
    "date (status+clock)": A1.FEATURE_GROUPS["date"], "clock only (hour/weekday/tz)": list(A1.CLOCK_FEATURES),
    "missingness flags only": [f for f in A1.MISSINGNESS_FEATURES if f in A1.SPEC], "subject shape (u_*)": A1.FEATURE_GROUPS["subject"],
    "all model features (full)": FEATURE_SETS["full"]}

def _src_cv(feats, rows, target):
    pred = np.full(n_train, -1)
    sub = np.asarray(rows)
    for tr, va in StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(sub)), target[sub], groups=GROUPS[sub]):
        tr, va = sub[tr], sub[va]
        if feats == "tokens":
            cv = CountVectorizer(binary=True, min_df=10, max_features=3000, token_pattern=r"\S+")
            Ztr = cv.fit_transform(templates[tr]); Zva = cv.transform(templates[va]); nb = MultinomialNB(alpha=0.5)
        else:
            pp = A1.HeaderPreprocessor(feats).fit(X_train.iloc[tr])
            Ztr, Zva = pp.transform(X_train.iloc[tr]), pp.transform(X_train.iloc[va])
            nb = CategoricalNB(alpha=1.0, min_categories=pp.n_categories_)
        pred[va] = nb.fit(Ztr, target[tr]).predict(Zva)
    return balanced_accuracy_score(target[sub], pred[sub])

ceas_sa = np.where(np.isin(src_train, ["ceas08", "spamassasin", "spamassassin"]))[0]
cs_target = (src_train == "ceas08").astype(int)
rows = []
t0 = time.time()
for name, feats in list(GROUP_DEFS.items()) + [("subject tokens (words)", "tokens")]:
    rows.append({"feature group": name, "4-source bal.acc (chance .25)": _src_cv(feats, ALL_IDX, src_codes),
                 "CEAS vs SpamAssassin bal.acc (chance .50)": _src_cv(feats, ceas_sa, cs_target)})
SRC_ID = pd.DataFrame(rows).set_index("feature group")
print(SRC_ID.round(3).to_string()); print(f"({time.time()-t0:.0f}s)")

                              4-source bal.acc (chance .25)  CEAS vs SpamAssassin bal.acc (chance .50)
feature group                                                                                         
sender (all s_*)                                      0.290                                      0.524
receiver (all r_*)                                    0.360                                      0.526
date (status+clock)                                   0.375                                      0.676
clock only (hour/weekday/tz)                          0.372                                      0.677
missingness flags only                                0.382                                      0.525
subject shape (u_*)                                   0.435                                      0.514
all model features (full)                             0.556                                      0.703
subject tokens (words)                                0.751              

## 7. Binary Nested Cross-Validation and Hyperparameter Optimization

Every candidate configuration in the 96-candidate grid is scored inside inner folds using threshold-independent metrics ($\text{AP}_{\text{Not Safe}}$, $\text{ROC-AUC}$, $\text{LogLoss}$) and $\text{F1}_{\text{Not Safe}}$ evaluated at a **fixed threshold of $0.5$** (no threshold tuning occurs during candidate selection, eliminating indirect inner-fold leakage):
$$\text{Score} = \frac{1}{4}\Big(\text{F1}_{\text{Not Safe}@0.5} + \text{AP}_{\text{Not Safe}} + \text{ROC-AUC} + \max(0, 1 - \text{LogLoss})\Big)$$
One-SE parsimony selects the simplest configuration within 1 standard error of the best inner score. **After** candidate selection, temperature scaling $T$ is fitted on the winning candidate's inner OOF predictions, and the operating thresholds (`argmax`, `f1_opt`, `macro_f1`, `recall@P>=0.60..0.80`) are tuned on those calibrated inner OOF predictions and inner labels only. The frozen $T$ and threshold are then applied to the untouched outer evaluation fold.

In [12]:
FLOORS = sorted(set(PRECISION_FLOORS) | {MIN_NOTSAFE_PRECISION})
OP_NAMES = ["argmax", "f1_opt", "macro_f1"] + [f"recall@P>={f:.2f}" for f in FLOORS]
CFG_KEYS = list(NESTED_GRID[0].keys())

def operating_points(P_cal, y):
    tab = A1.threshold_table(P_cal, y)
    ops = {
        "argmax": A1.pick_threshold(tab, "argmax"),
        "f1_opt": A1.pick_threshold(tab, "f1_opt"),
        "macro_f1": A1.pick_threshold(tab, "macro_f1"),
    }
    for f in FLOORS:
        ops[f"recall@P>={f:.2f}"] = A1.pick_threshold(tab, "recall_at_precision", f)
    return ops

def search_and_calibrate(idx, groups, grid, seed, purge=False, n_inner=N_INNER, ifolds=None, forbid=None):
    idx = np.asarray(idx)
    if forbid is not None:
        assert not (set(idx.tolist()) & set(np.asarray(forbid).tolist())), "outer-evaluation rows reached inner search"
    ifolds = make_group_folds(idx, groups, n_inner, seed, purge) if ifolds is None else ifolds
    if forbid is not None:
        used = np.concatenate([np.concatenate([tr, va]) for tr, va in ifolds])
        assert not (set(used.tolist()) & set(np.asarray(forbid).tolist()))
    cands = list(grid)
    tasks = []
    for cfg in cands:
        kw = cfg_to_agent_kwargs(cfg)
        for tr, va in ifolds:
            tasks.append((kw, _slim(X_train.iloc[tr], cfg), y_train[tr], _slim(X_train.iloc[va], cfg)))
    res = iter(run_tasks(tasks))
    rel = np.full(n_train, -1); rel[idx] = np.arange(len(idx))
    y_in = y_train[idx]; rows, Ps = [], []
    va_parts = np.concatenate([rel[va] for _, va in ifolds])
    assert len(va_parts) == len(idx) and np.array_equal(np.sort(va_parts), np.arange(len(idx))), "inner validation folds must partition idx exactly once"
    for ci, cfg in enumerate(cands):
        P = np.zeros((len(idx), 2))
        for tr, va in ifolds:
            P[rel[va]] = next(res)
        Ps.append(P)
        oof_pred_05 = np.full(len(idx), -1, dtype=int)
        per = []
        for tr, va in ifolds:
            r_va = rel[va]
            pred_va = (P[r_va, 1] >= 0.5).astype(int)
            oof_pred_05[r_va] = pred_va
            st = A1.binary_stats_at_preds(y_train[va], pred_va)
            ap_v = safe_ap(y_train[va] == 1, P[r_va, 1])
            auc_v = safe_roc_auc(y_train[va] == 1, P[r_va, 1])
            ll_v = float(log_loss(y_train[va], P[r_va], labels=[0, 1]))
            per.append((st["f1_1"], ap_v, auc_v, max(0.0, 1.0 - ll_v), ll_v))
        assert (oof_pred_05 >= 0).all(), "incomplete inner OOF coverage in oof_pred_05"
        pf = np.array(per, dtype=float)
        S = np.nanmean(pf[:, :4], axis=1); ok = np.isfinite(S)
        st_in = A1.binary_stats_at_preds(y_in, oof_pred_05)
        rows.append({
            **cfg, "cand": ci, "complexity": complexity(cfg),
            "score": float(S[ok].mean()),
            "score_se": float(S[ok].std(ddof=1) / np.sqrt(ok.sum())) if ok.sum() > 1 else 0.0,
            "f1_at_05": float(np.nanmean(pf[:, 0])),
            "ap": float(np.nanmean(pf[:, 1])),
            "roc_auc": float(np.nanmean(pf[:, 2])),
            "log_loss": float(log_loss(y_in, P, labels=[0, 1])),
            "P@0.5": st_in["prec"], "R@0.5": st_in["rec"],
        })
    R = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    best = R.iloc[0]
    eligible = R[R.score >= best.score - best.score_se].sort_values(["complexity", "score"], ascending=[True, False])
    chosen = eligible.iloc[0]; ci = int(chosen["cand"])
    T = A1.fit_temperature(Ps[ci], y_in)
    return dict(cfg=cands[ci], T=T, P_raw=Ps[ci], P_cal=A1.apply_temperature(Ps[ci], T), idx=idx, ifolds=ifolds,
                table=R, n_eligible=len(eligible), n_cands=len(cands))

def fit_one(cfg, tr):
    return A1.HeaderAgent(**cfg_to_agent_kwargs(cfg)).fit_features(X_train.iloc[tr], y_train[tr])

DATA_SHA = hashlib.sha256(pathlib.Path(DATA_PATH).read_bytes()).hexdigest() if "DATA_PATH" in globals() else ""
MODULE_SHA = hashlib.sha256(HEADER_SRC.encode("utf-8")).hexdigest()

def _cache_path(label, k, tr, grid, seed, purge, keep_models):
    key = json.dumps(["bin_v8_fixed05", label, k, int(seed), bool(purge), bool(keep_models), len(tr),
                      hashlib.sha256(np.asarray(tr, dtype=np.int64).tobytes()).hexdigest(),
                      [sorted(c.items()) for c in grid], DATA_SHA, MODULE_SHA, N_INNER, FLOORS], default=str)
    return OUT / "cache" / (hashlib.sha256(key.encode()).hexdigest()[:24] + ".joblib")

def run_nested(outer_folds, groups, grid, purge=False, label="", keep_models=False, seed_base=1000):
    t0 = time.time()
    P_out = np.zeros((n_train, 2)); covered = np.zeros(n_train, bool); fold_of = np.full(n_train, -1)
    preds = {n: np.full(n_train, -1) for n in OP_NAMES}; log, models, fold_T, fold_ops = [], [], [], []
    for k, (tr, va) in enumerate(outer_folds):
        assert not (set(tr) & set(va)), "outer train/eval overlap"
        cp = _cache_path(label, k, tr, grid, seed_base + k, purge, keep_models)
        if USE_CACHE and cp.exists():
            r = joblib.load(cp); sel_cfg, T_k, ops, Pcal, mdl, lrow = r["cfg"], r["T"], r["ops"], r["Pcal"], r["model"], r["log"]
            src = "cache"
        else:
            sel = search_and_calibrate(tr, groups, grid, seed=seed_base + k, purge=purge, forbid=va)
            ops = operating_points(sel["P_cal"], y_train[tr])
            mdl = fit_one(sel["cfg"], tr)
            Pcal = A1.apply_temperature(mdl.proba_features(X_train.iloc[va], calibrated=False), sel["T"])
            sel_cfg, T_k = sel["cfg"], sel["T"]
            lrow = {"fold": k, "n_train": len(tr), "n_eval": len(va),
                    **{c: sel_cfg[c] for c in ("feature_set", "alpha", "lik_scale", "group_weighting", "subject_tokens")},
                    "n_cands": sel["n_cands"], "n_within_1SE": sel["n_eligible"], "T": round(T_k, 3),
                    "t_f1_opt": round(float(ops["f1_opt"][0]), 3)}
            if USE_CACHE:
                joblib.dump(dict(cfg=sel_cfg, T=T_k, ops=ops, Pcal=Pcal, model=(mdl if keep_models else None), log=lrow), cp)
            src = "computed"
        P_out[va] = Pcal; covered[va] = True; fold_of[va] = k
        for n, (t_op, info) in ops.items():
            preds[n][va] = predict_from_P(Pcal, t_op)
        log.append(lrow); fold_T.append(T_k); fold_ops.append(ops)
        if keep_models:
            models.append(dict(model=mdl, T=T_k, ops=ops, va=va, tr=tr))
        el = time.time() - t0
        print(f"  [{label}] outer fold {k + 1}/{len(outer_folds)} ({src}, {el:.0f}s) -> {sel_cfg['feature_set']}, alpha={sel_cfg['alpha']}, lik={sel_cfg['lik_scale']}, tokens={sel_cfg['subject_tokens']}", flush=True)
    assert covered.all()
    return dict(P=P_out, preds=preds, log=pd.DataFrame(log), models=models, label=label, folds=outer_folds, fold_T=fold_T, fold_ops=fold_ops, fold_of=fold_of)

def nested_table(res, y=None, rows_mask=None, names=None):
    y = y_train if y is None else y; mk = np.ones(len(y), bool) if rows_mask is None else rows_mask
    return pd.DataFrame({n: metrics(y[mk], res["P"][mk], pred=res["preds"][n][mk]) for n in (OP_NAMES if names is None else names)}).T

print("binary nested machinery ready | rules:", OP_NAMES)

binary nested machinery ready | rules: ['argmax', 'f1_opt', 'macro_f1', 'recall@P>=0.60', 'recall@P>=0.65', 'recall@P>=0.70', 'recall@P>=0.75', 'recall@P>=0.80']


In [13]:
# ---- binary pipeline sanity tests (no-peeking, fold disjointness, metric parity)
_tr, _va = folds[0]
_small = STRESS_GRID[:4]
def _choose():
    s = search_and_calibrate(_tr, GROUPS, _small, seed=SEED + 999)
    o = operating_points(s["P_cal"], y_train[_tr])
    return s["cfg"], s["T"], {n: round(float(v[0]), 6) for n, v in o.items()}
ref = _choose()
y_bak = y_train.copy()
try:
    perm = np.random.default_rng(0).permutation(len(_va))
    y_train[_va] = y_bak[_va][perm]
    alt = _choose()
finally:
    y_train[:] = y_bak
assert ref[0] == alt[0] and abs(ref[1] - alt[1]) < 1e-12 and ref[2] == alt[2], "outer labels leaked!"
print("TEST 1 passed: selection, T, and thresholds independent of outer eval labels")

for k, (tr, va) in enumerate(folds):
    assert not (set(GROUPS[tr]) & set(GROUPS[va])), k
    assert (FOLD_ID[va] == k).all()
print("TEST 2 passed: outer folds sender-disjoint")

_rng = np.random.default_rng(1); _y = _rng.integers(0, 2, 500); _P = _rng.dirichlet(np.ones(2), 500)
_m = metrics(_y, _P)
assert abs(_m["macro_f1"] - f1_score(_y, (_P[:, 1] >= 0.5).astype(int), average="macro")) < 1e-12
assert abs(_m["ap"] - average_precision_score(_y, _P[:, 1])) < 1e-12
assert abs(_m["roc_auc"] - roc_auc_score(_y, _P[:, 1])) < 1e-12
print("TEST 3 passed: binary metrics() agrees with scikit-learn")

_fp = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED, purge=True)
_tr, _va = _fp[0]
try:
    search_and_calibrate(ALL_IDX, GROUPS, _small, seed=1, forbid=_va); raise SystemExit("forbid guard did not fire")
except AssertionError: pass
print("TEST 4 passed: forbid-guard rejects outer rows")

_cfg = STRESS_GRID[3]; _tr0, _va0 = folds[0]
_a = _fit_proba(cfg_to_agent_kwargs(_cfg), X_train.iloc[_tr0], y_train[_tr0], X_train.iloc[_va0])
_b = _fit_proba(cfg_to_agent_kwargs(_cfg), _slim(X_train.iloc[_tr0], _cfg), y_train[_tr0], _slim(X_train.iloc[_va0], _cfg))
assert np.array_equal(_a, _b)
print("TEST 5 passed: slimmed worker frames bit-identical")

TEST 1 passed: selection, T, and thresholds independent of outer eval labels
TEST 2 passed: outer folds sender-disjoint
TEST 3 passed: binary metrics() agrees with scikit-learn
TEST 4 passed: forbid-guard rejects outer rows
TEST 5 passed: slimmed worker frames bit-identical


In [14]:
t0 = time.time()
NEST_SENDER = run_nested(folds, GROUPS, NESTED_GRID, purge=False, label="sender-disjoint", keep_models=True, seed_base=1000)
print(f"\nnested sender-disjoint CV finished in {(time.time() - t0) / 60:.1f} min")
print(NEST_SENDER["log"].to_string(index=False))

DEC = decision_name()
full_report(y_train, NEST_SENDER["P"], 0.0, f"Nested CV, sender-disjoint, rule={DEC}", pred=NEST_SENDER["preds"][DEC])
full_report(y_train, NEST_SENDER["P"], 0.0, "Nested CV, sender-disjoint, argmax (0.5)", pred=NEST_SENDER["preds"]["argmax"])
print("\nPer-outer-fold stability:")
print(pd.DataFrame([metrics(y_train[va], NEST_SENDER["P"][va], pred=NEST_SENDER["preds"][DEC][va]) for _, va in folds])
      [["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc", "ap", "fpr", "log_loss"]].round(4).to_string())

  [sender-disjoint] outer fold 1/5 (computed, 42s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint] outer fold 2/5 (computed, 82s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint] outer fold 3/5 (computed, 122s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint] outer fold 4/5 (computed, 162s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint] outer fold 5/5 (computed, 200s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True

nested sender-disjoint CV finished in 3.3 min
 fold  n_train  n_eval  feature_set  alpha  lik_scale group_weighting  subject_tokens  n_cands  n_within_1SE     T  t_f1_opt
    0    27492    6764 compact_recv    1.0        1.0            flat            True       96             6 1.571      0.35
    1    27777    6479 compact_recv    1.0        1.0            flat            True       96             6 1.602      0.05
    2    27346    6910 compact_recv    1.0        1.0            flat      

## 8. Probability Calibration, Non-Saturation Check, and Operating Thresholds
Compares `argmax (0.5)`, `f1_opt`, `macro_f1`, and `recall@P>=0.60..0.80` (with FPR) on nested sender-disjoint predictions, plus reliability table and score non-saturation check (Agent 2 scores are nearly saturated, so Agent 1 should provide smooth, non-saturated `nb_score`).

In [15]:
tab = nested_table(NEST_SENDER)
tab["target_floor"] = [np.nan, np.nan, np.nan] + FLOORS
tab["precision_minus_floor"] = tab["precision"] - tab["target_floor"]
print("NESTED operating points (sender-disjoint):")
print(tab[["precision", "recall", "f1", "macro_f1", "accuracy", "fpr", "fnr", "target_floor", "precision_minus_floor"]].astype(float).round(4).to_string())
print("\nThreshold-free quality:",
      {k: round(float(v), 4) for k, v in metrics(y_train, NEST_SENDER["P"], pred=NEST_SENDER["preds"]["argmax"]).items() if k in ("ap", "roc_auc", "log_loss", "ece")})

p1_nest = NEST_SENDER["P"][:, 1]
print(f"\nNon-saturation check (nested calibrated nb_score): <0.01 = {(p1_nest < 0.01).mean():.1%}, >0.99 = {(p1_nest > 0.99).mean():.1%}, "
      f"std(nb_logit) = {A1.clipped_logit(p1_nest).std():.3f}")

print("\nPer-source results (nested, rule=%s):" % DEC)
print(by_source(y_train, NEST_SENDER["P"], NEST_SENDER["preds"][DEC], src_train)
      [["source", "n", "accuracy", "precision", "recall", "f1", "roc_auc", "ap", "fpr"]].round(4).to_string(index=False))

print("\nReliability table (nested calibrated P(Not Safe)):")
bins = pd.cut(p1_nest, [0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0], include_lowest=True)
print(pd.DataFrame({"n": pd.Series(y_train).groupby(bins, observed=True).size(),
                    "mean_p_not_safe": pd.Series(p1_nest).groupby(bins, observed=True).mean(),
                    "empirical_not_safe_rate": pd.Series(y_train).groupby(bins, observed=True).mean()}).round(3).to_string())

NESTED operating points (sender-disjoint):
                precision  recall      f1  macro_f1  accuracy     fpr     fnr  target_floor  precision_minus_floor
argmax             0.9398  0.9588  0.9492    0.9410    0.9421  0.0793  0.0412           NaN                    NaN
f1_opt             0.9413  0.9538  0.9475    0.9393    0.9404  0.0768  0.0462           NaN                    NaN
macro_f1           0.9505  0.9446  0.9476    0.9402    0.9411  0.0635  0.0554           NaN                    NaN
recall@P>=0.60     0.8403  0.9936  0.9105    0.8838    0.8899  0.2438  0.0064          0.60                 0.2403
recall@P>=0.65     0.8403  0.9936  0.9105    0.8838    0.8899  0.2438  0.0064          0.65                 0.1903
recall@P>=0.70     0.8403  0.9936  0.9105    0.8838    0.8899  0.2438  0.0064          0.70                 0.1403
recall@P>=0.75     0.8403  0.9936  0.9105    0.8838    0.8899  0.2438  0.0064          0.75                 0.0903
recall@P>=0.80     0.8403  0.9936  0.

## 9. Generalization Stress Tests: Subject-Template Dependence
Distinguishes ordinary sender-grouped CV from template-disjoint and sender-disjoint + template-purged evaluation.

In [16]:
TPL_RES = {}
G_TPL = np.array([("tpl:" + t) if t else f"emptytpl:{i}" for t, i in zip(templates, train_df["email_id"])], dtype=object)

if RUN_TEMPLATE_EXPERIMENTS:
    t0 = time.time()
    f_tpl = make_group_folds(ALL_IDX, G_TPL, N_SPLITS, SEED + 1)
    for tr, va in f_tpl: assert not (_tpl_set(tr) & _tpl_set(va))
    TPL_RES["template-disjoint"] = run_nested(f_tpl, G_TPL, STRESS_GRID, label="template-disjoint", seed_base=2000)

    f_pur = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED, purge=True)
    for tr, va in f_pur:
        assert not (set(GROUPS[tr]) & set(GROUPS[va])) and not (_tpl_set(tr) & _tpl_set(va))
    TPL_RES["sender-disjoint + template-purged"] = run_nested(f_pur, GROUPS, STRESS_GRID, purge=True, label="sender+template purged", seed_base=3000)
    print(f"template experiments finished in {(time.time() - t0) / 60:.1f} min")

    cmp_rows = {"sender-disjoint (headline)": metrics(y_train, NEST_SENDER["P"], pred=NEST_SENDER["preds"][DEC])}
    for k_, r_ in TPL_RES.items(): cmp_rows[k_] = metrics(y_train, r_["P"], pred=r_["preds"][DEC])
    CMP = pd.DataFrame(cmp_rows).T
    print(CMP[["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc", "ap", "fpr", "log_loss"]].astype(float).round(4).to_string())
else:
    print("RUN_TEMPLATE_EXPERIMENTS is False: skipped.")

  [template-disjoint] outer fold 1/5 (computed, 13s) -> full, alpha=1.0, lik=1.0, tokens=True
  [template-disjoint] outer fold 2/5 (computed, 25s) -> full, alpha=1.0, lik=1.0, tokens=True
  [template-disjoint] outer fold 3/5 (computed, 38s) -> full, alpha=1.0, lik=1.0, tokens=True
  [template-disjoint] outer fold 4/5 (computed, 50s) -> full, alpha=1.0, lik=1.0, tokens=True
  [template-disjoint] outer fold 5/5 (computed, 63s) -> compact_no_clock, alpha=1.0, lik=1.0, tokens=True
  [sender+template purged] outer fold 1/5 (computed, 8s) -> full, alpha=1.0, lik=1.0, tokens=True
  [sender+template purged] outer fold 2/5 (computed, 16s) -> full, alpha=1.0, lik=1.0, tokens=True
  [sender+template purged] outer fold 3/5 (computed, 24s) -> full, alpha=1.0, lik=1.0, tokens=True
  [sender+template purged] outer fold 4/5 (computed, 31s) -> full, alpha=1.0, lik=1.0, tokens=True
  [sender+template purged] outer fold 5/5 (computed, 39s) -> full, alpha=1.0, lik=1.0, tokens=True
template experiments fin

In [18]:

print("Seen vs unseen subject templates (nested sender-disjoint):")

seen = np.zeros(n_train, bool)

for k, (tr, va) in enumerate(folds):
    trt = _tpl_set(tr)
    seen[va] = [t in trt for t in templates[va]]

rows = {
    nm: metrics(
        y_train[mk],
        NEST_SENDER["P"][mk],
        pred=NEST_SENDER["preds"][DEC][mk]
    )
    for nm, mk in (
        ("seen template", seen),
        ("unseen or empty template", ~seen)
    )
}

print(
    pd.DataFrame(rows).T[
        ["n", "accuracy", "precision", "recall", "f1",
         "macro_f1", "roc_auc", "ap", "fpr", "log_loss", "ece"]
    ].astype(float).round(4).to_string()
)


def scenario(drop=(), keep_only=None):
    cols = [
        c for c in HEADER_COLS
        if (c in keep_only if keep_only else c not in drop)
    ]

    frame = train_df[HEADER_COLS].copy()

    for c in HEADER_COLS:
        if c not in cols:
            frame[c] = None

    Xs = A1.extract_features(frame)

    P = np.zeros((n_train, 2))
    pred = np.zeros(n_train, int)

    for m in NEST_SENDER["models"]:
        Pk = A1.apply_temperature(
            m["model"].proba_features(
                Xs.iloc[m["va"]],
                calibrated=False
            ),
            m["T"]
        )

        P[m["va"]] = Pk
        pred[m["va"]] = predict_from_P(
            Pk,
            m["ops"][DEC][0]
        )

    return P, pred


SCEN = {
    "baseline (all fields)": {},
    "no sender": dict(drop=("sender",)),
    "no receiver": dict(drop=("receiver",)),
    "no date": dict(drop=("date",)),
    "no subject": dict(drop=("subject",)),
    "sender only": dict(keep_only=("sender",)),
    "subject only": dict(keep_only=("subject",)),
    "date+receiver only": dict(keep_only=("date", "receiver"))
}


# FIX: explicitly unpack P and pred.
# Do not pass pred as the third positional argument,
# because the third parameter of metrics() is t_logit.
rob = {}

for name, kw in SCEN.items():
    P, pred = scenario(**kw)
    rob[name] = metrics(y_train, P, pred=pred)


print(f"\nField-removal robustness (nested outer models, rule={DEC}):")

print(
    pd.DataFrame({
        k: {
            "accuracy": v["accuracy"],
            "recall": v["recall"],
            "fpr": v["fpr"],
            "f1": v["f1"],
            "macro_f1": v["macro_f1"],
            "roc_auc": v["roc_auc"],
            "ap": v["ap"],
            "log_loss": v["log_loss"]
        }
        for k, v in rob.items()
    }).T.astype(float).round(4).to_string()
)


Seen vs unseen subject templates (nested sender-disjoint):
                                n  accuracy  precision  recall      f1  macro_f1  roc_auc      ap     fpr  log_loss     ece
seen template             19692.0    0.9821     0.9797  0.9896  0.9846    0.9816   0.9983  0.9987  0.0282    0.0530  0.0128
unseen or empty template  14564.0    0.8842     0.8862  0.9022  0.8941    0.8831   0.9555  0.9603  0.1372    0.2717  0.0197

Field-removal robustness (nested outer models, rule=f1_opt):
                       accuracy  recall     fpr      f1  macro_f1  roc_auc      ap  log_loss
baseline (all fields)    0.9404  0.9538  0.0768  0.9475    0.9393   0.9867  0.9896    0.1460
no sender                0.9169  0.9831  0.1685  0.9302    0.9138   0.9831  0.9862    0.2155
no receiver              0.9368  0.9655  0.1003  0.9451    0.9353   0.9858  0.9886    0.1561
no date                  0.9183  0.9786  0.1595  0.9310    0.9154   0.9824  0.9858    0.2100
no subject               0.7142  0.9169  0

## 10. Out-of-Distribution Diagnostics: Leave-One-Source-Out (LOSO)
* `CEAS_08` and `SpamAssasin` contain both classes (`0`, `1`): reports `accuracy, f1, ap, roc_auc, fpr` (comparable with Agent 2's LOSO accuracy: `0.8226` on held-out `CEAS_08`, `0.8753` on held-out `SpamAssasin`).
* `Nazario` and `Nigerian_Fraud` contain only class `1`: reports **recall only** (`precision, ap, roc_auc, fpr` are `NaN`/undefined).

In [19]:
def run_loso(grid, label, seed_base=5000, verbose=True):
    t0 = time.time(); rows, cms = [], {}
    P_all = np.zeros((n_train, 2)); preds = {n: np.full(n_train, -1) for n in OP_NAMES}; covered = np.zeros(n_train, bool)
    for j, s in enumerate(sorted(np.unique(src_train))):
        tr, va = np.where(src_train != s)[0], np.where(src_train == s)[0]
        assert not (set(src_train[tr]) & {s})
        if len(np.unique(y_train[tr])) < 2:
            rows.append(dict(held_out=s, note="remaining corpora lack a class")); continue
        sel = search_and_calibrate(tr, GROUPS, grid, seed=seed_base + j)
        ops = operating_points(sel["P_cal"], y_train[tr])
        m = fit_one(sel["cfg"], tr)
        P = A1.apply_temperature(m.proba_features(X_train.iloc[va], calibrated=False), sel["T"])
        P_all[va] = P; covered[va] = True
        for n, (t_op, _) in ops.items(): preds[n][va] = predict_from_P(P, t_op)
        for rule in dict.fromkeys(["argmax", DEC]):
            met = metrics(y_train[va], P, pred=preds[rule][va])
            if s in ("nazario", "nigerianfraud"):
                for undef_k in ("precision", "f1", "macro_f1", "roc_auc", "ap", "fpr", "log_loss"):
                    met[undef_k] = float("nan")
            rows.append(dict(held_out=s, rule=rule, **met, config=sel["cfg"]["feature_set"], tokens=sel["cfg"]["subject_tokens"], T=round(sel["T"], 2)))
            cms[(s, rule)] = confusion_matrix(y_train[va], preds[rule][va], labels=[0, 1])
    if verbose: print(f"  [{label}] LOSO finished in {(time.time() - t0) / 60:.1f} min")
    return dict(table=pd.DataFrame(rows), cms=cms, P=P_all, preds=preds, covered=covered, label=label)

LOSO_RES = {}
if RUN_LOSO:
    LOSO_RES["selected from stress grid"] = run_loso(STRESS_GRID, "LOSO", seed_base=5000)
    NOCLOCK_GRID = [c for c in STRESS_GRID if c["feature_set"] in ("no_clock", "compact_no_clock")]
    LOSO_RES["no-clock features only"] = run_loso(NOCLOCK_GRID, "LOSO no-clock", seed_base=6000)
    cols = ["held_out", "rule", "n", "n_safe", "n_not_safe", "accuracy", "precision", "recall", "f1", "ap", "roc_auc", "fpr", "config", "tokens", "T"]
    for k_, r_ in LOSO_RES.items():
        print(f"\n##### LOSO ({k_}) #####")
        print(r_["table"][cols].round(4).to_string(index=False))
    print("\nReference Agent 2 LOSO accuracy: CEAS_08 = 0.8226, SpamAssassin = 0.8753")
else:
    print("RUN_LOSO is False: skipped.")

  [LOSO] LOSO finished in 0.8 min
  [LOSO no-clock] LOSO finished in 0.5 min

##### LOSO (selected from stress grid) #####
     held_out   rule     n  n_safe  n_not_safe  accuracy  precision  recall     f1     ap  roc_auc    fpr           config  tokens    T
       ceas08 argmax 26949   12136       14813    0.6869     0.7863  0.5910 0.6748 0.7718   0.8136 0.1960 compact_no_clock    True 1.81
       ceas08 f1_opt 26949   12136       14813    0.6335     0.7914  0.4526 0.5758 0.7718   0.8136 0.1456 compact_no_clock    True 1.81
      nazario argmax  1062       0        1062    0.5198        NaN  0.5198    NaN    NaN      NaN    NaN             full    True 1.63
      nazario f1_opt  1062       0        1062    0.4642        NaN  0.4642    NaN    NaN      NaN    NaN             full    True 1.63
nigerianfraud argmax  2255       0        2255    0.8244        NaN  0.8244    NaN    NaN      NaN    NaN     compact_recv    True 1.62
nigerianfraud f1_opt  2255       0        2255    0.8098     

## 11. Systematic Feature-Group Ablation Study
Evaluates feature-group removal across sender-disjoint, template-disjoint, and LOSO protocols using binary metrics.

In [20]:
if RUN_ABLATION:
    t0 = time.time()
    keycols = ["feature_set", "alpha", "lik_scale", "group_weighting", "subject_tokens"]
    modal = NEST_SENDER["log"][keycols].astype(str).agg("|".join, axis=1).value_counts().index[0]
    base_cfg = next(c for c in NESTED_GRID if "|".join(str(c[k]) for k in keycols) == modal)
    base_feats = FEATURE_SETS[base_cfg["feature_set"]]
    VARIANTS = {
        "base (modal choice)": (base_feats, {}),
        "drop receiver group": ([f for f in base_feats if not f.startswith("r_")], {}),
        "drop clock (hour/weekday/tz)": ([f for f in base_feats if f not in A1.CLOCK_FEATURES], {}),
        "drop date group": ([f for f in base_feats if not f.startswith("d_")], {}),
        "drop sender group": ([f for f in base_feats if not f.startswith("s_")], {}),
        "drop subject shape (u_*)": ([f for f in base_feats if not f.startswith("u_")], {}),
        "drop subject tokens": (base_feats, {"subject_tokens": False}),
        "drop missingness flags": (base_feats, {"use_missingness_features": False}),
    }
    ab = {}
    for vi, (vname, (feats, extra)) in enumerate(VARIANTS.items()):
        FEATURE_SETS[f"abl:{vname}"] = list(feats)
        cfg = {**base_cfg, "feature_set": f"abl:{vname}", **extra}
        r1 = run_nested(folds, GROUPS, [cfg], label=f"abl {vi}", seed_base=7000)
        m1 = metrics(y_train, r1["P"], pred=r1["preds"][DEC])
        row = {"n_feats": len(feats), "sender: logloss": m1["log_loss"], "sender: ap": m1["ap"], "sender: roc_auc": m1["roc_auc"], "sender: macroF1": m1["macro_f1"]}
        if RUN_TEMPLATE_EXPERIMENTS:
            r2 = run_nested(f_tpl, G_TPL, [cfg], label=f"abl {vi} tpl", seed_base=7500)
            m2 = metrics(y_train, r2["P"], pred=r2["preds"][DEC])
            row.update({"template: logloss": m2["log_loss"], "template: ap": m2["ap"], "template: macroF1": m2["macro_f1"]})
        ab[vname] = row
    AB = pd.DataFrame(ab).T
    print(f"\nAblation results ({(time.time() - t0) / 60:.1f} min):")
    print(AB.astype(float).round(4).to_string())
else:
    print("RUN_ABLATION is False: skipped.")

  [abl 0] outer fold 1/5 (computed, 4s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0] outer fold 2/5 (computed, 9s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0] outer fold 3/5 (computed, 13s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0] outer fold 4/5 (computed, 17s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0] outer fold 5/5 (computed, 22s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0 tpl] outer fold 1/5 (computed, 4s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0 tpl] outer fold 2/5 (computed, 9s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0 tpl] outer fold 3/5 (computed, 13s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0 tpl] outer fold 4/5 (computed, 17s) -> abl:base (modal choice), alpha=1.0, lik=1.0, tokens=True
  [abl 0 tpl] outer fold 5/5 (computed, 22s) -> abl:base (modal choice), alpha=

## 12. Final Binary Model Training, Calibration Verification, and Freeze
Selects the final configuration on all `34,256` training rows using inner grouped CV, fits temperature $T$ and binary threshold `t_star` on inner OOF only, reports log-loss and ECE before/after calibration, verifies non-saturation and direction (`mean(nb_score | y=1) > mean(nb_score | y=0)`), and freezes `agent1`.

In [21]:
t0 = time.time()
final_sel = search_and_calibrate(ALL_IDX, GROUPS, FULL_GRID, seed=SEED, ifolds=folds)
if FINAL_CFG_OVERRIDE:
    ov = {**final_sel["cfg"], **FINAL_CFG_OVERRIDE}
    final_sel = search_and_calibrate(ALL_IDX, GROUPS, [ov], ifolds=folds)
FINAL_CFG = {k: (v.item() if hasattr(v, "item") else v) for k, v in final_sel["cfg"].items()}
assert np.array_equal(final_sel["idx"], ALL_IDX) and final_sel["P_cal"].shape == (n_train, 2)
print(f"final inner search: {final_sel['n_cands']} candidates, {final_sel['n_eligible']} within 1 SE, {(time.time() - t0) / 60:.1f} min")
print(final_sel["table"].head(10)[["feature_set", "alpha", "lik_scale", "group_weighting", "subject_tokens", "score", "score_se", "f1_at_05", "ap", "roc_auc", "log_loss"]].round(4).to_string())
print("\nFINAL_CFG:", FINAL_CFG)

T = final_sel["T"]; P_oof_cal = final_sel["P_cal"]; P_raw_oof = final_sel["P_raw"]
OPS_FINAL = operating_points(P_oof_cal, y_train)
T_THRESH, THRESH_INFO = OPS_FINAL[DEC]
if not THRESH_INFO["feasible"]:
    print(f"!! precision floor infeasible on training OOF -> fell back to {THRESH_INFO['mode_used']}")

ll_raw = log_loss(y_train, P_raw_oof, labels=[0, 1])
ll_cal = log_loss(y_train, P_oof_cal, labels=[0, 1])
ece_raw = ece_binary(P_raw_oof, y_train)
ece_cal = ece_binary(P_oof_cal, y_train)
print(f"temperature T = {T:.3f} | threshold_logit = {T_THRESH:.3f} (prob = {THRESH_INFO['threshold_prob']:.3f}) | rule = {DEC}")
print(f"log-loss raw={ll_raw:.4f} -> calibrated={ll_cal:.4f} | ECE raw={ece_raw:.4f} -> calibrated={ece_cal:.4f}")

# Direction & non-saturation check on training OOF
m0 = float(P_oof_cal[y_train == 0, 1].mean())
m1 = float(P_oof_cal[y_train == 1, 1].mean())
assert m1 > m0, f"Direction check failed: mean nb_score for y=1 ({m1:.4f}) <= y=0 ({m0:.4f})"
print(f"Direction check PASS: mean(nb_score|y=0)={m0:.4f} < mean(nb_score|y=1)={m1:.4f}")
print(f"Non-saturation check (training OOF): <0.01={(P_oof_cal[:,1]<0.01).mean():.1%}, >0.99={(P_oof_cal[:,1]>0.99).mean():.1%}")

inproto = metrics(y_train, P_oof_cal, pred=predict_from_P(P_oof_cal, T_THRESH))
nested_m = metrics(y_train, NEST_SENDER["P"], pred=NEST_SENDER["preds"][DEC])

agent1 = fit_one(FINAL_CFG, ALL_IDX)
agent1.set_temperature(T); agent1.set_threshold_logit(T_THRESH)
print("\nFrozen binary Agent 1. Training rows:", n_train, "| val/test labels used: 0")

final inner search: 96 candidates, 6 within 1 SE, 0.9 min
        feature_set  alpha  lik_scale group_weighting  subject_tokens   score  score_se  f1_at_05      ap  roc_auc  log_loss
0      compact_recv    1.0        1.0            flat            True  0.9404    0.0048    0.9492  0.9897   0.9869    0.1646
1      compact_recv    0.5        1.0            flat            True  0.9404    0.0048    0.9492  0.9897   0.9869    0.1646
2      compact_recv    1.0        1.0       inv_group            True  0.9388    0.0049    0.9484  0.9895   0.9865    0.1693
3      compact_recv    0.5        1.0       inv_group            True  0.9388    0.0049    0.9484  0.9895   0.9865    0.1693
4              full    1.0        1.0            flat            True  0.9388    0.0049    0.9487  0.9901   0.9871    0.1710
5              full    0.5        1.0            flat            True  0.9388    0.0049    0.9487  0.9901   0.9871    0.1710
6              full    1.0        1.0       inv_group            Tr

## 13. Search Space Ablation: Full Factorial vs. Constrained Grid
Compares the full 96-candidate nested search against the reduced grid (`lik_scale=1.0`).

In [22]:
if RUN_REDUCED_COMPARISON and NESTED_GRID_MODE == "full":
    NEST_REDUCED = run_nested(folds, GROUPS, REDUCED_GRID, label="sender-disjoint REDUCED", seed_base=1000)
    NEST_FULL = NEST_SENDER
    def fold_metrics(res, rule):
        return pd.DataFrame([metrics(y_train[va], res["P"][va], pred=res["preds"][rule][va]) for _, va in folds])
    print("Per-fold comparison (full vs reduced):")
    print(pd.concat({"reduced": fold_metrics(NEST_REDUCED, DEC)[["macro_f1", "f1", "recall", "precision", "ap", "roc_auc", "log_loss"]],
                     "full": fold_metrics(NEST_FULL, DEC)[["macro_f1", "f1", "recall", "precision", "ap", "roc_auc", "log_loss"]]}, axis=1).round(4).to_string())
else:
    NEST_REDUCED = None
    print("Reduced-grid comparison skipped.")

  [sender-disjoint REDUCED] outer fold 1/5 (computed, 15s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint REDUCED] outer fold 2/5 (computed, 31s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint REDUCED] outer fold 3/5 (computed, 46s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint REDUCED] outer fold 4/5 (computed, 61s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
  [sender-disjoint REDUCED] outer fold 5/5 (computed, 77s) -> compact_recv, alpha=1.0, lik=1.0, tokens=True
Per-fold comparison (full vs reduced):
   reduced                                                        full                                                   
  macro_f1      f1  recall precision      ap roc_auc log_loss macro_f1      f1  recall precision      ap roc_auc log_loss
0   0.9346  0.9452  0.9441    0.9463  0.9902  0.9864   0.1485   0.9346  0.9452  0.9441    0.9463  0.9902  0.9864   0.1485
1   0.9515  0.9618  0.9584    0.9653  0.9927  0.9894   

## 14. Optional: Strictly Cross-Fitted Training OOF
Since Agent 3 fits on `val` rows (`7,340`) using `transformer_scores.csv` and `agent1_scores.csv`, training OOF generation is **optional** (`GENERATE_TRAIN_OOF = False` by default).

In [23]:
OOF_RES, OOF_DESC = None, "skipped (Agent 3 fits on val)"
if GENERATE_TRAIN_OOF:
    if OOF_PURGE_TEMPLATES:
        f_oof = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED, purge=True)
        OOF_RES = run_nested(f_oof, GROUPS, FULL_GRID, purge=True, label="agent3 OOF (template-purged)", seed_base=11000)
        OOF_DESC = "template-purged, sender-grouped"
    else:
        OOF_RES = NEST_SENDER; OOF_DESC = "sender-grouped"
    print("Optional train OOF finished:", OOF_DESC)
else:
    print("GENERATE_TRAIN_OOF is False: skipped (Agent 3 fits on val).")

GENERATE_TRAIN_OOF is False: skipped (Agent 3 fits on val).


## 15. Validation and Test Set Reporting (Frozen Protocol)
Evaluates the frozen binary model on `val` (for reporting only; never used for tuning). `test` remains untouched by default (`REPORT_ON_TEST = False`) and is not a pristine hold-out.

In [24]:
def report_split(name, frame, Xf):
    yb = get_binary_target(frame)
    P = agent1.proba_features(Xf, calibrated=True)
    pred = predict_from_P(P, agent1.threshold_logit)
    full_report(yb, P, agent1.threshold_logit, f"{name} (frozen binary model, rule={DEC}, reporting only)", pred=pred)
    full_report(yb, P, 0.0, f"{name} argmax (0.5)")
    print(f"\n{name}: per-source")
    print(by_source(yb, P, pred, frame["source"].map(norm_source).to_numpy())
          [["source", "n", "accuracy", "precision", "recall", "f1", "roc_auc", "ap", "fpr"]].round(4).to_string(index=False))
    return metrics(yb, P, pred=pred)

VAL_M = report_split("VALIDATION", val_df, X_val) if REPORT_ON_VAL else None
TEST_M = report_split("TEST (NOT pristine)", test_df, X_test) if REPORT_ON_TEST else None
if VAL_M is not None:
    MCOLS = ["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc", "ap", "fpr", "fnr", "log_loss", "ece"]
    print("\nValidation vs nested sender-disjoint estimate:")
    print(pd.DataFrame({"nested sender-disjoint (train)": {k: nested_m[k] for k in MCOLS},
                        "validation (frozen)": {k: VAL_M[k] for k in MCOLS}}).T.astype(float).round(4).to_string())


=== VALIDATION (frozen binary model, rule=f1_opt, reporting only) ===
          precision  recall      f1  support
safe         0.9452  0.9429  0.9441     3204
not_safe     0.9558  0.9577  0.9568     4136
confusion (rows=true, cols=pred):
               pred_safe  pred_not_safe
true_safe           3021            183
true_not_safe        175           3961
{'accuracy': 0.9512, 'precision': 0.9558, 'recall': 0.9577, 'f1': 0.9568, 'macro_f1': 0.9504, 'roc_auc': 0.9901, 'ap': 0.9921, 'fpr': 0.0571, 'fnr': 0.0423, 'log_loss': 0.125, 'ece': 0.0088}

=== VALIDATION argmax (0.5) ===
          precision  recall      f1  support
safe         0.9462  0.9385  0.9423     3204
not_safe     0.9527  0.9587  0.9557     4136
confusion (rows=true, cols=pred):
               pred_safe  pred_not_safe
true_safe           3007            197
true_not_safe        171           3965
{'accuracy': 0.9499, 'precision': 0.9527, 'recall': 0.9587, 'f1': 0.9557, 'macro_f1': 0.949, 'roc_auc': 0.9901, 'ap': 0.9921, '

## 16. Agent 1 Independent Export Contract

Agent 1 operates independently of Agent 2 and produces its own validated outputs for downstream integration in Agent 3:
1. **`agent1_scores.csv`** (contract file): exact columns `email_id, split, nb_score` for all `val` and `test` rows.
2. **`agent1_scores_detailed.csv`** (detailed file): adds `p_safe, nb_logit, pred_class, header_completeness, insufficient_evidence`.

Cross-agent alignment and merging with Agent 2 (`transformer_scores.csv`) are performed later in **Agent 3**.

In [25]:
def export_frame(det, ids, split):
    out = det.reset_index(drop=True).copy()
    out.insert(0, "split", split)
    out.insert(0, "email_id", np.asarray(ids))
    return out

det_val  = agent1.detailed_features(X_val).reset_index(drop=True)
det_test = agent1.detailed_features(X_test).reset_index(drop=True)
detailed_df = pd.concat([export_frame(det_val, val_df["email_id"], "val"),
                         export_frame(det_test, test_df["email_id"], "test")], ignore_index=True)

# Agent 1 export integrity assertions
assert list(detailed_df.columns) == ["email_id", "split", "p_safe", "nb_score", "nb_logit", "pred_class", "header_completeness", "insufficient_evidence"]
assert np.isfinite(detailed_df[["p_safe", "nb_score", "nb_logit"]].to_numpy(float)).all()
assert detailed_df["p_safe"].between(0.0, 1.0).all()
assert detailed_df["nb_score"].between(0.0, 1.0).all()
assert np.allclose(detailed_df["p_safe"] + detailed_df["nb_score"], 1.0, atol=1e-6)
assert detailed_df["pred_class"].isin([0, 1]).all()
assert not detailed_df["email_id"].isna().any() and detailed_df["email_id"].is_unique
assert not detailed_df.duplicated(["email_id", "split"]).any()
assert not set(detailed_df["email_id"]) & set(train_df["email_id"]), "train ids must not appear in val/test export"
assert detailed_df.groupby("split").size().to_dict() == {"val": len(val_df), "test": len(test_df)}

# Contract file: email_id, split, nb_score
scores_df = detailed_df[["email_id", "split", "nb_score"]].copy()
assert list(scores_df.columns) == ["email_id", "split", "nb_score"]
assert len(scores_df) == len(val_df) + len(test_df)

# Verify exact one-to-one alignment with source dataset val/test IDs and splits
expected_vt = df[df["split"].isin(["val", "test"])][["email_id", "split"]]
merged_chk = scores_df[["email_id", "split"]].merge(expected_vt, on="email_id", suffixes=("_out", "_src"), validate="one_to_one")
assert len(merged_chk) == len(scores_df) and (merged_chk["split_out"] == merged_chk["split_src"]).all(), "export IDs/splits mismatch source dataset"

scores_df.to_csv(OUT / "agent1_scores.csv", index=False)
detailed_df.to_csv(OUT / "agent1_scores_detailed.csv", index=False)
print("Exported independently:", OUT / "agent1_scores.csv", "and", OUT / "agent1_scores_detailed.csv")
print("Split counts:", scores_df["split"].value_counts().to_dict())
print(scores_df.head(5).to_string(index=False))

Exported independently: agent1_binary_outputs/agent1_scores.csv and agent1_binary_outputs/agent1_scores_detailed.csv
Split counts: {'test': 7343, 'val': 7340}
 email_id split  nb_score
        1   val  0.995654
        8   val  0.000152
       19   val  0.001326
       29   val  0.932171
       39   val  0.991829


## 17. Artifact Serialization and Standalone Inference API
Saves `agent1_binary_model.joblib` and `agent1_binary_meta.json`, exposes `score_email(sender, receiver, date, subject) -> float`, and verifies reload parity in a fresh subprocess.

In [26]:
MODEL_PATH, META_PATH = OUT / "agent1_binary_model.joblib", OUT / "agent1_binary_meta.json"
MCOLS_SAVE = ["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc", "ap", "fpr", "fnr", "log_loss", "ece"]
f4 = lambda m: {k: (None if (isinstance(v, float) and np.isnan(v)) else float(v)) for k, v in m.items() if k in MCOLS_SAVE}

bundle = dict(
    agent=agent1, version=A1.__version__, class_order=CLASS_NAMES, selected_config=FINAL_CFG,
    temperature=float(T), decision_rule=DEC, threshold_logit=float(T_THRESH),
    threshold_info={k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in THRESH_INFO.items()},
    training_rows=int(n_train), cv_grouping=GROUP_LEVEL, seed=SEED, n_splits=N_SPLITS, n_inner=N_INNER,
    nested_grid_mode=NESTED_GRID_MODE, nested_candidates=len(NESTED_GRID), quick_search=bool(QUICK_SEARCH),
    export_columns=list(scores_df.columns), detailed_columns=list(detailed_df.columns),
    nested_sender_disjoint_metrics=f4(nested_m), validation_metrics=(f4(VAL_M) if VAL_M else None),
    libraries=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, scipy=scipy.__version__),
    dataset_sha256=DATA_SHA, module_sha256=MODULE_SHA,
)
joblib.dump(bundle, MODEL_PATH)
META_PATH.write_text(json.dumps({k: v for k, v in bundle.items() if k != "agent"}, indent=2, default=str), encoding="utf-8")

def score_email(sender, receiver, date, subject):
    # Standalone inference API returning calibrated P(Not Safe) in [0, 1].
    return float(A1.score_email(agent1, {"sender": sender, "receiver": receiver, "date": date, "subject": subject}))

loaded = joblib.load(MODEL_PATH)["agent"]
sample = val_df[HEADER_COLS].head(200)
p_mem, p_load = agent1.predict_proba(sample).to_numpy(), loaded.predict_proba(sample).to_numpy()
assert np.allclose(p_mem, p_load, atol=0)
assert np.allclose(p_mem[:, 1], scores_df[scores_df.split == "val"]["nb_score"].head(200).to_numpy(), atol=1e-6)
r0 = sample.iloc[0].to_dict()
assert abs(score_email(r0["sender"], r0["receiver"], r0["date"], r0["subject"]) - float(p_mem[0, 1])) < 1e-12

# Fresh subprocess parity check
rows50 = sample.head(50).astype(object).where(sample.head(50).notna(), None).to_dict("records")
(OUT / "_parity_input.json").write_text(json.dumps(rows50, default=str), encoding="utf-8")
code_ = ("import sys, json; sys.path.insert(0, %r); import joblib, agent1_header as A; b = joblib.load(%r); "
         "rows = json.load(open(%r)); p = b['agent'].predict_proba(rows).to_numpy().tolist(); json.dump(p, open(%r, 'w'))"
         ) % (str(OUT.resolve()), str(MODEL_PATH.resolve()), str((OUT / "_parity_input.json").resolve()), str((OUT / "_parity_output.json").resolve()))
proc = subprocess.run([sys.executable, "-c", code_], capture_output=True, text=True)
if proc.returncode: raise RuntimeError(proc.stderr)
p_fresh = np.array(json.loads((OUT / "_parity_output.json").read_text()))
assert np.allclose(p_fresh, p_mem[:50], atol=1e-12)
for f_ in ("_parity_input.json", "_parity_output.json"): (OUT / f_).unlink()
print("Saved:", sorted(p.name for p in OUT.iterdir()))
print("Binary serialization, score_email() API, and fresh-process parity: PASS")

Saved: ['__pycache__', 'agent1_binary_meta.json', 'agent1_binary_model.joblib', 'agent1_header.py', 'agent1_scores.csv', 'agent1_scores_detailed.csv', 'cache']
Binary serialization, score_email() API, and fresh-process parity: PASS


## 18. Scientific Scope, Binary Limitations, and Integration Conclusions

* **Binary Target:** Agent 1 now trains natively on `0 = Safe`, `1 = Not Safe` with `K=2` priors, likelihoods, subject-token NB, log-odds temperature scaling, and a single documented threshold.
* **Calibration & Non-Saturation:** Unlike DistilBERT's nearly saturated `text_score`, Agent 1 provides smooth, non-saturated `nb_score` and clipped `nb_logit` for Agent 3's Logistic Regression stacker.
* **Header-Only Ceiling & Corpus Shift:** As shown by LOSO (`CEAS_08`, `SpamAssasin`, `Nazario`, `Nigerian_Fraud`), header signals alone are sensitive to corpus metadata habits and cannot detect phishing sent from compromised legitimate accounts. Agent 1 serves as a lightweight header-triage signal combined with Agent 2 by Agent 3.

In [27]:
tiers = {
    "1. inner-CV selection (optimistic)": inproto,
    "2. NESTED sender-disjoint, FULL 96-cand (headline)": nested_m,
}
if NEST_REDUCED is not None:
    tiers["2b. nested sender-disjoint, REDUCED"] = metrics(y_train, NEST_REDUCED["P"], pred=NEST_REDUCED["preds"][DEC])
for k_, r_ in (TPL_RES.items() if RUN_TEMPLATE_EXPERIMENTS else []):
    tiers[f"3. nested {k_}"] = metrics(y_train, r_["P"], pred=r_["preds"][DEC])
if VAL_M is not None:
    tiers["4. validation (frozen, reporting only)"] = VAL_M
if TEST_M is not None:
    tiers["5. test (NOT pristine)"] = TEST_M

print(f"Binary Summary, decision rule = {DEC}")
print(pd.DataFrame(tiers).T[["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc", "ap", "fpr", "log_loss", "ece"]].astype(float).round(4).to_string())
print("\nExports for Agent 3: agent1_scores.csv (contract), agent1_scores_detailed.csv, agent1_binary_model.joblib")

Binary Summary, decision rule = f1_opt
                                                    accuracy  precision  recall      f1  macro_f1  roc_auc      ap     fpr  log_loss     ece
1. inner-CV selection (optimistic)                    0.9423     0.9423  0.9563  0.9492    0.9413   0.9867  0.9896  0.0756    0.1460  0.0080
2. NESTED sender-disjoint, FULL 96-cand (headline)    0.9404     0.9413  0.9538  0.9475    0.9393   0.9867  0.9896  0.0768    0.1460  0.0087
2b. nested sender-disjoint, REDUCED                   0.9404     0.9413  0.9538  0.9475    0.9393   0.9867  0.9896  0.0768    0.1460  0.0087
3. nested template-disjoint                           0.9368     0.9260  0.9650  0.9451    0.9353   0.9813  0.9838  0.0996    0.1704  0.0160
3. nested sender-disjoint + template-purged           0.8708     0.9389  0.8244  0.8779    0.8704   0.9682  0.9735  0.0693    0.2409  0.0491
4. validation (frozen, reporting only)                0.9512     0.9558  0.9577  0.9568    0.9504   0.9901  0.9921 